# Ground-truth Generators

- $v_1 = \partial_t$
- $v_2 = u\partial_u$
- $v_3 = e^{-k^2 t}[k^{-2}\partial_x - \partial_y]$
- $v_4 = t\partial_x + \partial_y - \frac{1}{2}(y+k^2x)u\partial_u$
- $v_5 = \partial_x$
- $v_6 = e^{k^2 t}[k^{-2}\partial_x + \partial_y - yu\partial_u]$

$v_1, v_3, v_5$ are the SDE symmetry generators

# SDE Symmetry

##Imports & Downloads

In [ ]:
!pip install dm-haiku

In [ ]:

import math
import functools
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import optax

jax.config.update("jax_enable_x64", True)

import itertools
from functools import partial

import matplotlib.pyplot as plt

Array = jnp.ndarray

In [ ]:
print(jnp.array([0.]).dtype)   # should print float64


## Neural SDE surrogate

In [ ]:
# @title Neural SDE surrogate for Example 4 (Gaeta & Rodríguez Quintero 1999, Sec.6):
#   dx = y dt
#   dy = -k2 y dt + sqrt(2 k2) dW_t
#
# Runs as-is in Google Colab (JAX + optax).
#
# Diagnostics included:
#   ✅ dt-grid check, dX consistency check
#   ✅ noise-only check:
#        eps_x = Δx - y dt    (should be ~0, var~0)
#        eps_y = Δy + k2 y dt (should have var ~ (2*k2)*dt)
#   ✅ plots: sample trajectories + increment hist/scatter + training + learned drift/sigma grids
#
# Notes:
#   - We use a diagonal noise model for training (independent Gaussian per component):
#       Δx ~ N(fx dt,  (σx^2) dt),  Δy ~ N(fy dt, (σy^2) dt)
#     True σx = 0, true σy = sqrt(2*k2). We keep σ_min>0 for numerical stability.

import math
from dataclasses import dataclass

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

try:
    import optax
except Exception:
    raise RuntimeError("This cell expects optax to be available in Colab.")

jax.config.update("jax_enable_x64", True)

# -------------------------------------------------------------------
# 0. Config
# -------------------------------------------------------------------

@dataclass
class CFG:
    # Ground-truth SDE (Example 4)
    k2: float = 1.0  # positive constant (called k^2 in the paper)

    # Time grid
    T: float = 5.0
    dt: float = 0.01
    n_traj: int = 2048

    # Initial condition distribution for (x0,y0)
    xy0_mode: str = "uniform"   # "point", "normal", "uniform", "mixture_points", "grid_points"
    x0_value: float = 0.0
    y0_value: float = 0.0
    xy0_mean: tuple = (0.0, 0.0)
    xy0_std: tuple = (1.0, 1.0)
    x0_low: float = -0.5
    x0_high: float = 0.5
    y0_low: float = -0.5
    y0_high: float = 0.5
    xy0_points: tuple = ((-1.0, -1.0), (-1.0, 1.0), (0.0, 0.0), (1.0, -1.0), (1.0, 1.0))

    # NN + training
    hidden: int = 64
    steps: int = 10000
    batch_size: int = 4096
    lr: float = 3e-3
    weight_decay: float = 1e-6

    # noise floor
    sigma_min: float = 1e-3

    # optional: do we allow the net to learn σx or keep it ~ sigma_min?
    learn_sigma_x: bool = True

    # Optional: warning thresholds
    warn_if_exploding: bool = True
    explode_std_threshold: float = 50.0

cfg = CFG(k2=1.0, T=2.0, dt=0.01, xy0_mode="uniform",
          x0_low=-0.5, x0_high=0.5, y0_low=-0.5, y0_high=0.5,
          learn_sigma_x=True)
key_main = jax.random.PRNGKey(0)

# -------------------------------------------------------------------
# 1. Data generation: Euler–Maruyama for Example 4
# -------------------------------------------------------------------

def sample_xy0(k_init, cfg: CFG):
    if cfg.xy0_mode == "point":
        x0 = jnp.full((cfg.n_traj, 1), cfg.x0_value, dtype=jnp.float64)
        y0 = jnp.full((cfg.n_traj, 1), cfg.y0_value, dtype=jnp.float64)
        return jnp.concatenate([x0, y0], axis=1)  # (n_traj, 2)

    if cfg.xy0_mode == "normal":
        mx, my = cfg.xy0_mean
        sx, sy = cfg.xy0_std
        z = jax.random.normal(k_init, (cfg.n_traj, 2), dtype=jnp.float64)
        return jnp.array([mx, my], dtype=jnp.float64) + z * jnp.array([sx, sy], dtype=jnp.float64)

    if cfg.xy0_mode == "uniform":
        kx, ky = jax.random.split(k_init, 2)
        x0 = jax.random.uniform(kx, (cfg.n_traj, 1), minval=cfg.x0_low, maxval=cfg.x0_high, dtype=jnp.float64)
        y0 = jax.random.uniform(ky, (cfg.n_traj, 1), minval=cfg.y0_low, maxval=cfg.y0_high, dtype=jnp.float64)
        return jnp.concatenate([x0, y0], axis=1)

    if cfg.xy0_mode == "mixture_points":
        pts = jnp.array(cfg.xy0_points, dtype=jnp.float64)  # (K,2)
        idx = jax.random.randint(k_init, (cfg.n_traj,), 0, pts.shape[0])
        return pts[idx]

    if cfg.xy0_mode == "grid_points":
        pts = jnp.array(cfg.xy0_points, dtype=jnp.float64)
        K = pts.shape[0]
        reps = int(math.ceil(cfg.n_traj / K))
        return jnp.tile(pts, (reps, 1))[:cfg.n_traj, :]

    raise ValueError(f"Unknown cfg.xy0_mode: {cfg.xy0_mode}")

def make_ex4_data(key, cfg: CFG):
    """
    Euler–Maruyama (batched over trajectories):
      x_{n+1} = x_n + y_n dt
      y_{n+1} = y_n + (-k2 y_n) dt + sqrt(2 k2) dW_n,  dW_n ~ N(0, dt)
    Returns:
      t  : (N+1,)
      XY : (n_traj, N+1, 2) with columns [x,y]
    """
    dt = cfg.dt
    N = int(cfg.T / dt)
    t = jnp.linspace(0.0, cfg.T, N + 1)

    k_init, k_noise = jax.random.split(key, 2)
    xy0 = sample_xy0(k_init, cfg)  # (n_traj, 2)

    dW = jax.random.normal(k_noise, (cfg.n_traj, N), dtype=jnp.float64) * math.sqrt(dt)  # (n_traj, N)

    sqrt_2k2 = math.sqrt(2.0 * cfg.k2)

    def step(xy, dWn):
        # xy: (n_traj, 2), dWn: (n_traj,)
        x = xy[:, 0]
        y = xy[:, 1]
        x1 = x + y * dt
        y1 = y + (-cfg.k2 * y) * dt + sqrt_2k2 * dWn
        xy1 = jnp.stack([x1, y1], axis=1)  # (n_traj, 2)
        return xy1, xy1

    _, xys = jax.lax.scan(step, xy0, dW.T)  # xys: (N, n_traj, 2)

    XY = jnp.concatenate([xy0[:, None, :], jnp.swapaxes(xys, 0, 1)], axis=1)  # (n_traj, N+1, 2)
    return t, XY


t, XY = make_ex4_data(key_main, cfg)
print("Data shapes: t =", t.shape, ", XY =", XY.shape)

t_np = np.asarray(jax.device_get(t))
XY_np = np.asarray(jax.device_get(XY))
x_np = XY_np[..., 0]
y_np = XY_np[..., 1]

# Plot a subset of trajectories
n_plot = min(20, cfg.n_traj)
stride = max(1, len(t_np) // 500)

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
for i in range(n_plot):
    plt.plot(t_np[::stride], x_np[i, ::stride], lw=1, alpha=0.6)
plt.xlabel("t"); plt.ylabel("x(t)")
plt.title("Sample trajectories: x(t)")
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
for i in range(n_plot):
    plt.plot(t_np[::stride], y_np[i, ::stride], lw=1, alpha=0.6)
plt.xlabel("t"); plt.ylabel("y(t)")
plt.title("Sample trajectories: y(t)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(5,5))
for i in range(n_plot):
    plt.plot(x_np[i, ::stride], y_np[i, ::stride], lw=1, alpha=0.6)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Phase plot (x,y)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n[RAW PATH DATA]")
print("cfg.dt =", cfg.dt)
print("t[:10] =", t_np[:10])
print("t[1]-t[0] =", t_np[1] - t_np[0], "  (should be ~ cfg.dt)")
print("first traj (x,y) first 5:\n", XY_np[0, :5, :])
print("x stats: mean =", x_np.mean(), "std =", x_np.std(), "min =", x_np.min(), "max =", x_np.max())
print("y stats: mean =", y_np.mean(), "std =", y_np.std(), "min =", y_np.min(), "max =", y_np.max())

if cfg.warn_if_exploding and (max(x_np.std(), y_np.std()) > cfg.explode_std_threshold):
    print("\n[WARNING] Large trajectory std. This can happen for long T / large noise.")
    print("          Reduce T or dt, or shrink init spread for a calmer demo.")

# -------------------------------------------------------------------
# 2. Build training dataset: (t_n, x_n, y_n) -> Δ[x,y]_n
# -------------------------------------------------------------------

def build_increment_dataset(t, XY):
    XY_n = XY[:, :-1, :]      # (n_traj, N, 2)
    XY_np1 = XY[:, 1:, :]     # (n_traj, N, 2)
    dXY = XY_np1 - XY_n       # (n_traj, N, 2)

    t_n = jnp.broadcast_to(t[:-1][None, :, None], (XY_n.shape[0], XY_n.shape[1], 1))
    inp = jnp.concatenate([t_n, XY_n], axis=-1)  # (n_traj, N, 3) = [t,x,y]

    return inp.reshape(-1, 3), dXY.reshape(-1, 2)

X_raw, dX = build_increment_dataset(t, XY)
print("\nIncrement dataset shapes: X_raw =", X_raw.shape, ", dX =", dX.shape)

# ------------------- Validation CHECKS + NOISE-ONLY CHECKS -------------------
print("\n[VALIDATION CHECKS]")
print("dt from grid (float):", float(t[1] - t[0]), " ; cfg.dt:", cfg.dt)

dXY_direct = XY[:, 1:, :] - XY[:, :-1, :]               # (n_traj,N,2)
dX_mat = dX.reshape(XY.shape[0], -1, 2)                  # (n_traj,N,2)
max_abs_err = float(jnp.max(jnp.abs(dX_mat - dXY_direct)))
print("max|dX - (XY[:,1:]-XY[:,:-1])| =", max_abs_err)

# Noise-only residuals
x_n = XY[:, :-1, 0]
y_n = XY[:, :-1, 1]
dx = dXY_direct[..., 0]
dy = dXY_direct[..., 1]

eps_x = dx - (y_n * cfg.dt)                 # should be ~0
eps_y = dy - ((-cfg.k2 * y_n) * cfg.dt)     # should be ~ sqrt(2k2)*dW

print("\n[NOISE-ONLY CHECKS]")
print("eps_x = Δx - y dt: mean =", float(jnp.mean(eps_x)), " var =", float(jnp.var(eps_x)),
      "  (should be ~0, ~0)")
print("eps_y = Δy + k2 y dt: mean =", float(jnp.mean(eps_y)),
      " var =", float(jnp.var(eps_y)),
      "  (should be ~ (2*k2)*dt =", (2.0 * cfg.k2) * cfg.dt, ")")

print("\n[INFO] var(Δx) is NOT σ^2 dt here; Δx is driven by y(t) through drift.")
print("var(Δx) =", float(jnp.var(dx)), " ; var(Δy) =", float(jnp.var(dy)))

# --- Histograms of increments ---
dX_np = np.asarray(jax.device_get(dX))
dx_np = dX_np[:, 0]
dy_np = dX_np[:, 1]

max_hist = 200_000
rng = np.random.default_rng(0)
if dx_np.size > max_hist:
    idx = rng.choice(dx_np.size, size=max_hist, replace=False)
    dx_plot = dx_np[idx]
    dy_plot = dy_np[idx]
else:
    dx_plot = dx_np
    dy_plot = dy_np

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.hist(dx_plot, bins=80, density=True, alpha=0.8)
plt.xlabel(r"$\Delta x$"); plt.ylabel("density")
plt.title(r"Increment distribution $\Delta x$")
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.hist(dy_plot, bins=80, density=True, alpha=0.8)
plt.xlabel(r"$\Delta y$"); plt.ylabel("density")
plt.title(r"Increment distribution $\Delta y$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Scatter: Δx vs y_n should align ~ y dt ---
Xraw_np = np.asarray(jax.device_get(X_raw))
y_n_flat = Xraw_np[:, 2]
dx_flat = dx_np

max_scatter = 50_000
if dx_flat.size > max_scatter:
    idx = rng.choice(dx_flat.size, size=max_scatter, replace=False)
    y_sc = y_n_flat[idx]
    dx_sc = dx_flat[idx]
else:
    y_sc = y_n_flat
    dx_sc = dx_flat

plt.figure(figsize=(6, 4))
plt.scatter(y_sc, dx_sc, s=2, alpha=0.25, label="samples")
y_line = np.linspace(y_sc.min(), y_sc.max(), 200)
plt.plot(y_line, y_line * cfg.dt, lw=2, label=r"theory: $y\,\Delta t$")
plt.xlabel(r"$y_n$")
plt.ylabel(r"$\Delta x_n$")
plt.title(r"Check: $\Delta x$ vs $y_n$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 3. Normalization (z-score) for inputs (t,x,y)
# -------------------------------------------------------------------

class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

def fit_normalizer(X):
    return Normalizer(jnp.mean(X, axis=0), jnp.std(X, axis=0))

in_norm = fit_normalizer(X_raw)
X = in_norm(X_raw)

print("\n[NORMALIZATION CHECK]")
print("Input mean ~", np.asarray(jax.device_get(X.mean(0))), "std ~", np.asarray(jax.device_get(X.std(0))))

# -------------------------------------------------------------------
# 4. MLP model: (t,x,y) -> (f̂x,f̂y, g_x,g_y) with σ̂ = softplus(g)+σ_min
# -------------------------------------------------------------------

def glorot(key, fan_in, fan_out):
    lim = math.sqrt(6.0 / (fan_in + fan_out))
    return jax.random.uniform(key, (fan_in, fan_out), minval=-lim, maxval=lim)

def init_mlp_params(key, sizes):
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (m, n) in zip(keys, zip(sizes[:-1], sizes[1:])):
        params.append({"W": glorot(k, m, n), "b": jnp.zeros((n,), dtype=jnp.float64)})
    return params

def mlp_forward(params, x, activation="tanh"):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            if activation == "tanh":
                h = jnp.tanh(h)
            elif activation == "relu":
                h = jax.nn.relu(h)
            else:
                raise ValueError(f"Unknown activation {activation}")
    return h

def f_sigma_hat(params, txy_norm, activation="tanh", sigma_min=1e-3, learn_sigma_x=True):
    out = mlp_forward(params, txy_norm, activation=activation)  # (...,4)
    f_hat = out[..., 0:2]  # (...,2)
    g = out[..., 2:4]      # (...,2)
    sigma_hat = jax.nn.softplus(g) + sigma_min
    if not learn_sigma_x:
        sigma_hat = sigma_hat.at[..., 0].set(sigma_min)  # force σx ~ sigma_min
    return f_hat, sigma_hat

key_main, k_model = jax.random.split(key_main, 2)
params_sde = init_mlp_params(k_model, sizes=[3, cfg.hidden, cfg.hidden, 4])

# -------------------------------------------------------------------
# 4b. Standard evaluation wrappers for downstream pipeline compatibility
# -------------------------------------------------------------------

def _broadcast_txy(t, x, y):
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.asarray(y, dtype=jnp.float64)
    t_b, x_b, y_b = jnp.broadcast_arrays(t, x, y)
    return jnp.stack([t_b, x_b, y_b], axis=-1)  # (...,3)

def eval_sde_surrogate_txy(params, txy, *, activation="tanh"):
    """
    txy: (...,3) = [t,x,y] in *raw* coordinates (NOT normalized).
    Returns:
      f_hat    : (...,2)  drift components [f_x, f_y]
      sigma_hat: (...,2)  diagonal diffusion magnitudes [sigma_x, sigma_y]
    """
    txy = jnp.asarray(txy, dtype=jnp.float64)
    txy_norm = in_norm(txy)
    return f_sigma_hat(params, txy_norm, activation=activation,
                       sigma_min=cfg.sigma_min, learn_sigma_x=cfg.learn_sigma_x)

def eval_sde_surrogate(params, t, x, y, *, activation="tanh"):
    """
    Convenience wrapper: broadcast (t,x,y) -> evaluate surrogate.
    Returns (f_hat, sigma_hat) with shapes (...,2), (...,2).
    """
    return eval_sde_surrogate_txy(params, _broadcast_txy(t, x, y), activation=activation)

# -------------------------------------------------------------------
# 5. Increment-based diagonal Gaussian NLL loss
# -------------------------------------------------------------------

def make_increment_loss(dt, sigma_min, weight_decay, learn_sigma_x=True):
    def loss_fn(params, xb, dxb):
        f_hat, sigma_hat = f_sigma_hat(params, xb, activation="tanh",
                                       sigma_min=sigma_min, learn_sigma_x=learn_sigma_x)
        mu = f_hat * dt                      # (...,2)
        var = (sigma_hat ** 2) * dt          # (...,2)

        residual = dxb - mu                  # (...,2)
        ell = (residual ** 2) / (2.0 * (var + 1e-12)) + 0.5 * jnp.log(var + 1e-12)
        nll = jnp.mean(jnp.sum(ell, axis=-1))

        def l2_tree(p):
            return sum([jnp.sum(v**2) for v in jax.tree_util.tree_leaves(p)])

        reg = weight_decay * l2_tree(params)
        return nll + reg
    return loss_fn

loss_fn = make_increment_loss(cfg.dt, cfg.sigma_min, cfg.weight_decay, learn_sigma_x=cfg.learn_sigma_x)

# -------------------------------------------------------------------
# 6. Training loop
# -------------------------------------------------------------------

optimizer = optax.adam(cfg.lr)
opt_state = optimizer.init(params_sde)
rng_np = np.random.default_rng(0)

@jax.jit
def train_step(params, opt_state, xb, dxb):
    def _loss(p):
        return loss_fn(p, xb, dxb)
    val, grads = jax.value_and_grad(_loss)(params)
    updates, opt_state_new = optimizer.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, val

def sample_minibatch(X, dX, batch_size):
    N = X.shape[0]
    if batch_size >= N:
        idx = np.arange(N)
    else:
        idx = rng_np.choice(N, size=batch_size, replace=False)
    return X[idx], dX[idx]

print_every = 100
loss_history = []

for step in range(1, cfg.steps + 1):
    xb, dxb = sample_minibatch(X, dX, cfg.batch_size)
    xb = jnp.asarray(xb)
    dxb = jnp.asarray(dxb)

    params_sde, opt_state, loss_val = train_step(params_sde, opt_state, xb, dxb)
    loss_float = float(loss_val)
    loss_history.append(loss_float)

    if step % print_every == 0 or step == 1 or step == cfg.steps:
        print(f"step {step:5d}/{cfg.steps} | NLL+reg = {loss_float:.6e}")

print("\nTraining finished.")

# -------------------------------------------------------------------
# 7. Plot training loss curve
# -------------------------------------------------------------------

steps_arr = np.arange(1, cfg.steps + 1)
plt.figure(figsize=(6, 4))
plt.plot(steps_arr, loss_history, lw=2)
plt.xlabel("Training step")
plt.ylabel("NLL + reg")
plt.title("Training loss for neural SDE surrogate (Example 4)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 8. Diagnostics: learned f̂ and σ̂ on a grid + compare to truth
# -------------------------------------------------------------------

def eval_model(params, t_vals, x_vals, y_vals):
    TT, XX, YY = jnp.meshgrid(t_vals, x_vals, y_vals, indexing="ij")
    TXY = jnp.stack([TT.ravel(), XX.ravel(), YY.ravel()], axis=-1)
    TXY_norm = in_norm(TXY)
    f_hat, sigma_hat = f_sigma_hat(params, TXY_norm, activation="tanh",
                                   sigma_min=cfg.sigma_min, learn_sigma_x=cfg.learn_sigma_x)
    shape = TT.shape
    return TT, XX, YY, f_hat.reshape(*shape, 2), sigma_hat.reshape(*shape, 2)

t_eval = jnp.linspace(0.0, cfg.T, 21)
x_eval = jnp.linspace(-2.0, 2.0, 21)
y_eval = jnp.linspace(-2.0, 2.0, 21)

TT, XX, YY, F_hat, Sig_hat = eval_model(params_sde, t_eval, x_eval, y_eval)

# Truth
F_true_x = YY
F_true_y = -cfg.k2 * YY
Sig_true_x = 0.0
Sig_true_y = math.sqrt(2.0 * cfg.k2)

print("\nGround-truth:")
print("  f_x(t,x,y)=y, f_y(t,x,y)=-k2*y with k2 =", cfg.k2)
print("  sigma_x=0, sigma_y=sqrt(2*k2) =", Sig_true_y)
print("\nLearned summaries over eval grid:")
print("  mean σ̂_x:", float(Sig_hat[...,0].mean()), "   std σ̂_x:", float(Sig_hat[...,0].std()))
print("  mean σ̂_y:", float(Sig_hat[...,1].mean()), "   std σ̂_y:", float(Sig_hat[...,1].std()))
print("  mean |f̂_x - y|:", float(jnp.abs(F_hat[...,0] - F_true_x).mean()))
print("  mean |f̂_y + k2 y|:", float(jnp.abs(F_hat[...,1] - F_true_y).mean()))

# Visualize at a fixed x-slice (since truth doesn't depend on x)
ix = len(x_eval)//2
Fhx = np.asarray(F_hat[:, ix, :, 0])     # (t,y)
Fhy = np.asarray(F_hat[:, ix, :, 1])
Shx = np.asarray(Sig_hat[:, ix, :, 0])
Shy = np.asarray(Sig_hat[:, ix, :, 1])

y_eval_np = np.asarray(y_eval)
t_eval_np = np.asarray(t_eval)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

im00 = axes[0,0].imshow(Shx, origin="lower",
                        extent=(float(y_eval.min()), float(y_eval.max()),
                                float(t_eval.min()), float(t_eval.max())),
                        aspect="auto")
axes[0,0].set_title(r"$\hat{\sigma}_x(t,\cdot,y)$ (x-slice)")
axes[0,0].set_xlabel("y"); axes[0,0].set_ylabel("t")
fig.colorbar(im00, ax=axes[0,0], shrink=0.9)

im01 = axes[0,1].imshow(Shy, origin="lower",
                        extent=(float(y_eval.min()), float(y_eval.max()),
                                float(t_eval.min()), float(t_eval.max())),
                        aspect="auto")
axes[0,1].set_title(r"$\hat{\sigma}_y(t,\cdot,y)$ (x-slice)")
axes[0,1].set_xlabel("y"); axes[0,1].set_ylabel("t")
fig.colorbar(im01, ax=axes[0,1], shrink=0.9)

im10 = axes[1,0].imshow(Fhx, origin="lower",
                        extent=(float(y_eval.min()), float(y_eval.max()),
                                float(t_eval.min()), float(t_eval.max())),
                        aspect="auto")
axes[1,0].set_title(r"$\hat{f}_x(t,\cdot,y)$ (x-slice)")
axes[1,0].set_xlabel("y"); axes[1,0].set_ylabel("t")
fig.colorbar(im10, ax=axes[1,0], shrink=0.9)

im11 = axes[1,1].imshow(Fhy, origin="lower",
                        extent=(float(y_eval.min()), float(y_eval.max()),
                                float(t_eval.min()), float(t_eval.max())),
                        aspect="auto")
axes[1,1].set_title(r"$\hat{f}_y(t,\cdot,y)$ (x-slice)")
axes[1,1].set_xlabel("y"); axes[1,1].set_ylabel("t")
fig.colorbar(im11, ax=axes[1,1], shrink=0.9)

plt.show()

# 1D slice at mid time: compare learned vs truth as a function of y
mid_t = len(t_eval)//2
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(y_eval_np, Fhx[mid_t, :], lw=2, label=r"$\hat f_x$")
plt.plot(y_eval_np, y_eval_np, lw=2, label=r"true $y$")
plt.xlabel("y"); plt.ylabel("drift x-component")
plt.title(f"Drift f_x slice at t≈{float(t_eval[mid_t]):.2f}")
plt.grid(True, alpha=0.3); plt.legend()

plt.subplot(1,2,2)
plt.plot(y_eval_np, Fhy[mid_t, :], lw=2, label=r"$\hat f_y$")
plt.plot(y_eval_np, -cfg.k2*y_eval_np, lw=2, label=r"true $-k2\,y$")
plt.xlabel("y"); plt.ylabel("drift y-component")
plt.title(f"Drift f_y slice at t≈{float(t_eval[mid_t]):.2f}")
plt.grid(True, alpha=0.3); plt.legend()

plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 9. Export trained objects to globals
# -------------------------------------------------------------------

globals().update({
    "cfg": cfg,
    "params_sde": params_sde,
    "in_norm_sde": in_norm,
    "loss_history_sde": list(zip(steps_arr, loss_history)),
    # downstream-friendly API:
    "eval_sde_surrogate": eval_sde_surrogate,
    "eval_sde_surrogate_txy": eval_sde_surrogate_txy,
})

print("\nExported: params_sde, in_norm_sde, cfg, loss_history_sde, eval_sde_surrogate, eval_sde_surrogate_txy.")


In [ ]:
# === Animations: drift and diffusion vs y as time evolves =====================
# Requirements in scope from training cell:
#   - params_sde   : trained SDE surrogate params
#   - in_norm_sde  : Normalizer for (t,x,y)
#   - f_sigma_hat  : (params, txy_norm) -> (f_hat(.,2), sigma_hat(.,2))
#   - cfg          : config with at least cfg.k2 (and optionally cfg.sigma_min, cfg.T)
#   - t            : (Nt,) time grid used to simulate data (if missing, we'll rebuild from cfg)

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ----------------- helpers ----------------------------------------------------
sigma_min = getattr(cfg, "sigma_min", 1e-3)

def eval_f_sigma_vs_y(params, in_norm, t0, x_slice=0.0, y_min=-2.0, y_max=2.0, n_points=250):
    """
    Evaluate learned f̂(t0, x_slice, y) and σ̂(t0, x_slice, y) on a 1D y-grid.
    Returns:
      y_vals: (N,)
      f_x   : (N,)
      f_y   : (N,)
      sig_x : (N,)
      sig_y : (N,)
    """
    y_vals = jnp.linspace(y_min, y_max, n_points)
    t_vals = jnp.full_like(y_vals, t0)
    x_vals = jnp.full_like(y_vals, x_slice)
    TXY = jnp.stack([t_vals, x_vals, y_vals], axis=-1)     # (N,3)
    TXY_norm = in_norm(TXY)
    f_hat, sigma_hat = f_sigma_hat(params, TXY_norm,
                                   activation="tanh",
                                   sigma_min=sigma_min,
                                   learn_sigma_x=getattr(cfg, "learn_sigma_x", True))
    f_x = f_hat[:, 0]
    f_y = f_hat[:, 1]
    sig_x = sigma_hat[:, 0]
    sig_y = sigma_hat[:, 1]
    return (np.asarray(y_vals),
            np.asarray(f_x), np.asarray(f_y),
            np.asarray(sig_x), np.asarray(sig_y))

# Time grid for animation frames
if "t" in globals():
    t_array = np.asarray(t)
    t_start, t_end = float(t_array[0]), float(t_array[-1])
else:
    t_start, t_end = 0.0, float(getattr(cfg, "T", 5.0))
t_frames = np.linspace(t_start, t_end, 120)   # frames

# y-range for plots
y_min, y_max = -2.0, 2.0
x_slice = 0.0  # slice since truth does not depend on x

# Ground-truth references
k2 = float(getattr(cfg, "k2", 1.0))
sig_true_x = 0.0
sig_true_y = float(np.sqrt(2.0 * k2))

# ================== 1) Drift animation (fx, fy vs y) =========================
fig_drift, ax_drift = plt.subplots(figsize=(7, 4))
line_fx, = ax_drift.plot([], [], lw=2, label=r"learned $\hat f_x(t,x_*,y)$")
line_fy, = ax_drift.plot([], [], lw=2, label=r"learned $\hat f_y(t,x_*,y)$")

# truth lines (for current y-grid we redraw each frame to match grid)
truth_fx, = ax_drift.plot([], [], "k--", lw=1.5, label=r"true $f_x=y$")
truth_fy, = ax_drift.plot([], [], "k-.", lw=1.5, label=r"true $f_y=-k^2 y$")

ax_drift.set_xlim(y_min, y_max)
ax_drift.set_xlabel("y")
ax_drift.set_ylabel("drift")
title_drift = ax_drift.set_title("")
ax_drift.grid(True, alpha=0.3)
ax_drift.legend(loc="upper left")

def init_drift():
    line_fx.set_data([], [])
    line_fy.set_data([], [])
    truth_fx.set_data([], [])
    truth_fy.set_data([], [])
    title_drift.set_text("")
    return line_fx, line_fy, truth_fx, truth_fy, title_drift

def update_drift(frame_idx):
    t0 = float(t_frames[frame_idx])
    y_vals, fx, fy, _, _ = eval_f_sigma_vs_y(params_sde, in_norm_sde, t0,
                                             x_slice=x_slice, y_min=y_min, y_max=y_max)
    line_fx.set_data(y_vals, fx)
    line_fy.set_data(y_vals, fy)

    # ground-truth curves
    truth_fx.set_data(y_vals, y_vals)
    truth_fy.set_data(y_vals, -k2 * y_vals)

    title_drift.set_text(rf"Drift at $t={t0:.2f}$ (slice $x_*={x_slice:g}$)")
    return line_fx, line_fy, truth_fx, truth_fy, title_drift

anim_drift = FuncAnimation(
    fig_drift,
    update_drift,
    init_func=init_drift,
    frames=len(t_frames),
    interval=80,
    blit=True,
)

plt.close(fig_drift)
display(HTML(anim_drift.to_jshtml()))

# ================== 2) Diffusion animation (sigx, sigy vs y) =================
fig_diff, ax_diff = plt.subplots(figsize=(7, 4))
line_sigx, = ax_diff.plot([], [], lw=2, label=r"learned $\hat\sigma_x(t,x_*,y)$")
line_sigy, = ax_diff.plot([], [], lw=2, label=r"learned $\hat\sigma_y(t,x_*,y)$")

# truth lines (constants)
ax_diff.axhline(sig_true_x, color="k", linestyle="--", lw=1.5,
                label=rf"true $\sigma_x={sig_true_x:g}$")
ax_diff.axhline(sig_true_y, color="k", linestyle="-.", lw=1.5,
                label=rf"true $\sigma_y=\sqrt{{2k^2}}={sig_true_y:.3g}$")

ax_diff.set_xlim(y_min, y_max)
ax_diff.set_xlabel("y")
ax_diff.set_ylabel("diffusion (diag)")
title_diff = ax_diff.set_title("")
ax_diff.grid(True, alpha=0.3)
ax_diff.legend(loc="upper left")

# y-lims: try to auto-reasonable around true σy
pad = 0.2 * max(1.0, sig_true_y)
ax_diff.set_ylim(min(-0.05, sig_true_x - pad), sig_true_y + pad)

def init_diff():
    line_sigx.set_data([], [])
    line_sigy.set_data([], [])
    title_diff.set_text("")
    return line_sigx, line_sigy, title_diff

def update_diff(frame_idx):
    t0 = float(t_frames[frame_idx])
    y_vals, _, _, sigx, sigy = eval_f_sigma_vs_y(params_sde, in_norm_sde, t0,
                                                 x_slice=x_slice, y_min=y_min, y_max=y_max)
    line_sigx.set_data(y_vals, sigx)
    line_sigy.set_data(y_vals, sigy)
    title_diff.set_text(rf"Diffusion at $t={t0:.2f}$ (slice $x_*={x_slice:g}$)")
    return line_sigx, line_sigy, title_diff

anim_diff = FuncAnimation(
    fig_diff,
    update_diff,
    init_func=init_diff,
    frames=len(t_frames),
    interval=80,
    blit=True,
)

plt.close(fig_diff)
display(HTML(anim_diff.to_jshtml()))


## Surrogate apply helper

In [ ]:
# Helper: evaluate surrogate drift and diffusion at (t, x, y)
# Requires: params_sde, in_norm_sde, f_sigma_hat, cfg, jax, jnp

def surrogate_f_sigma(params_sde, in_norm_sde, t, x, y, activation="tanh"):
    """
    Evaluate the learned 2D SDE surrogate (f̂, σ̂) at (t, x, y).

    Inputs:
      t : scalar / array, broadcastable with x and y
      x : scalar / array, broadcastable with t and y
      y : scalar / array, broadcastable with t and x

    Returns:
      f_hat     : (..., 2) drift components [f_x, f_y]
      sigma_hat : (..., 2) diagonal diffusion components [sigma_x, sigma_y]
                 (i.e., Sigma = diag(sigma_x, sigma_y))
    """
    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)
    y_arr = jnp.asarray(y, dtype=jnp.float64)

    # Broadcast to a common shape
    t_b, x_b, y_b = jnp.broadcast_arrays(t_arr, x_arr, y_arr)

    # Stack into (B, 3) with B = number of (t,x,y) points
    txy = jnp.stack([t_b.ravel(), x_b.ravel(), y_b.ravel()], axis=-1)  # (B, 3)

    # Normalize using the training-time normalizer
    txy_norm = in_norm_sde(txy)

    # Forward through neural surrogate
    f_hat_flat, sigma_hat_flat = f_sigma_hat(
        params_sde,
        txy_norm,
        activation=activation,
        sigma_min=cfg.sigma_min,
        learn_sigma_x=getattr(cfg, "learn_sigma_x", True),
    )  # f_hat_flat: (B,2), sigma_hat_flat: (B,2)

    # Reshape back to broadcasted shape
    out_shape = t_b.shape
    f_hat = f_hat_flat.reshape((*out_shape, 2))
    sigma_hat = sigma_hat_flat.reshape((*out_shape, 2))

    return f_hat, sigma_hat


## Data Generation & plots

In [ ]:
# Generate data from the learned neural 2D SDE surrogate for Stage-2 training
# Requires: surrogate_f_sigma, params_sde, in_norm_sde, cfg, jax, jnp, np, matplotlib

from dataclasses import dataclass
import math

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

@dataclass
class CFGGen:
    T: float = cfg.T           # reuse same horizon
    dt: float = cfg.dt         # reuse same time step
    n_traj: int = 256          # fewer trajectories than Stage 1

cfg_gen = CFGGen()

def _init_xy0(key, cfg_gen: CFGGen):
    """
    Try to reuse Stage-1 IC sampler if available; otherwise fall back to zeros.
    Returns: xy0 (n_traj, 2)
    """
    if "sample_xy0" in globals() and "cfg" in globals():
        try:
            xy0 = sample_xy0(key, cfg)  # expects cfg.n_traj; so override if needed
            if xy0.shape[0] != cfg_gen.n_traj:
                # re-sample with a temporary cfg copy if sizes mismatch
                cfg_tmp = type(cfg)(**{**cfg.__dict__, "n_traj": cfg_gen.n_traj})
                xy0 = sample_xy0(key, cfg_tmp)
            return xy0
        except Exception:
            pass

    # fallback
    return jnp.zeros((cfg_gen.n_traj, 2), dtype=jnp.float64)

def simulate_surrogate_paths(key, params_sde, in_norm_sde, cfg_gen: CFGGen):
    """
    Simulate paths from the neural 2D SDE surrogate using Euler–Maruyama:
        [x;y]_{n+1} = [x;y]_n + f̂(t_n, x_n, y_n) dt + diag(σ̂_x, σ̂_y) * dW_n
    where dW_n ~ N(0, dt I_2).

    Returns:
      t_sim  : (N+1,) time grid
      XY_sim : (n_traj, N+1, 2) paths with columns [x,y]
    """
    dt = float(cfg_gen.dt)
    T  = float(cfg_gen.T)
    n_traj = int(cfg_gen.n_traj)

    N = int(T / dt)
    t_sim = jnp.linspace(0.0, T, N + 1, dtype=jnp.float64)

    # Initial condition (x0,y0)
    key, k_init = jax.random.split(key, 2)
    xy0 = _init_xy0(k_init, cfg_gen)  # (n_traj, 2)

    # Pre-generate independent Gaussian noise for both components: dW ~ N(0, dt)
    key, k_noise = jax.random.split(key, 2)
    dW = jax.random.normal(
        k_noise,
        shape=(N, n_traj, 2),
        dtype=jnp.float64,
    ) * math.sqrt(dt)  # already scaled by sqrt(dt)

    def step(xy_t, inputs):
        k_idx, t_curr = inputs

        # xy_t: (n_traj, 2)
        x_t = xy_t[:, 0]
        y_t = xy_t[:, 1]
        t_vec = jnp.full((n_traj,), t_curr, dtype=jnp.float64)

        # Evaluate surrogate drift/diffusion at (t, x, y)
        f_hat, sigma_hat = surrogate_f_sigma(
            params_sde, in_norm_sde, t_vec, x_t, y_t
        )  # (n_traj,2), (n_traj,2)

        dW_t = dW[k_idx]  # (n_traj, 2)

        # Euler–Maruyama with diagonal diffusion (elementwise multiply)
        xy_next = xy_t + f_hat * dt + sigma_hat * dW_t
        return xy_next, xy_next

    idxs = jnp.arange(N, dtype=jnp.int32)
    _, xy_hist = jax.lax.scan(step, xy0, (idxs, t_sim[:-1]))  # (N, n_traj, 2)

    # Stack initial condition and transpose to (n_traj, N+1, 2)
    XY_sim = jnp.concatenate([xy0[None, :, :], xy_hist], axis=0)  # (N+1, n_traj, 2)
    XY_sim = jnp.swapaxes(XY_sim, 0, 1)                           # (n_traj, N+1, 2)

    return t_sim, XY_sim

# Simulate surrogate paths
key_main, key_surr = jax.random.split(key_main)
t_surr, XY_surr = simulate_surrogate_paths(key_surr, params_sde, in_norm_sde, cfg_gen)

print("Surrogate sim shapes: t_surr =", t_surr.shape, ", XY_surr =", XY_surr.shape)


# Initialize t_s7 and xy_paths_s7
if ("t_s7" not in globals()) or ("xy_paths_s7" not in globals()):
    if ("t_surr" in globals()) and ("XY_surr" in globals()):
        t_src = globals()["t_surr"]
        XY_src = globals()["XY_surr"]
    else:
        t_src = globals().get("t", None)
        XY_src = globals().get("XY", None)
    assert t_src is not None and XY_src is not None, "Need (t_surr, XY_surr) or (t, XY) for 2D S7."

    n_traj_use = int(min(64, XY_src.shape[0]))
    Np1_use    = int(min(401, XY_src.shape[1]))
    t_s7 = jnp.asarray(t_src[:Np1_use], dtype=jnp.float64)
    xy_paths_s7 = jnp.asarray(XY_src[:n_traj_use, :Np1_use, :], dtype=jnp.float64)  # (n_traj, T, 2)
    globals().update({"t_s7": t_s7, "xy_paths_s7": xy_paths_s7})

# Build flattened (t, x, y) dataset for generator training
def build_txy_dataset(t, XY):
    """
    Flatten paths into a cloud of (t_n, x_n, y_n) points:
      t  : (N+1,)
      XY : (n_traj, N+1, 2)
    Returns:
      TX_gen : (B, 3) where B = n_traj * (N+1), columns [t,x,y]
    """
    n_traj, Np1, _ = XY.shape
    t_b = jnp.broadcast_to(t[None, :], (n_traj, Np1))  # (n_traj, N+1)

    t_flat = t_b.reshape(-1, 1)
    x_flat = XY[..., 0].reshape(-1, 1)
    y_flat = XY[..., 1].reshape(-1, 1)

    TX_gen = jnp.concatenate([t_flat, x_flat, y_flat], axis=1)  # (B,3)
    return TX_gen

TX_gen = build_txy_dataset(t_surr, XY_surr)
print("TX_gen shape (flattened (t,x,y) points):", TX_gen.shape)

# Convenience views
x_surr = XY_surr[..., 0]
y_surr = XY_surr[..., 1]

# Export to globals for later use
globals().update({
    "t_surr": t_surr,
    "XY_surr": XY_surr,
    "x_surr": x_surr,
    "y_surr": y_surr,
    "TX_gen": TX_gen,
    "cfg_gen": cfg_gen,
})

# Quick plots: sample a few surrogate paths
n_plot = min(20, x_surr.shape[0])
idx_plot = np.linspace(0, x_surr.shape[0] - 1, n_plot, dtype=int)

t_np = np.asarray(t_surr)
x_np = np.asarray(x_surr)
y_np = np.asarray(y_surr)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for idx in idx_plot:
    plt.plot(t_np, x_np[idx], alpha=0.6)
plt.xlabel("t"); plt.ylabel("x(t)")
plt.title("Sample x(t) trajectories (surrogate)")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for idx in idx_plot:
    plt.plot(t_np, y_np[idx], alpha=0.6)
plt.xlabel("t"); plt.ylabel("y(t)")
plt.title("Sample y(t) trajectories (surrogate)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 5))
for idx in idx_plot:
    plt.plot(x_np[idx], y_np[idx], alpha=0.6)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Phase plot (x,y) from surrogate")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Generator Neural Net Setup

In [ ]:
# Generator neural net setup: τ_i(t) and ξ_i(t,x,y) for 2D SDE symmetries (+ β_i for FP)
# Requires:
#   - jax, jax.numpy as jnp
#   - init_mlp_params, mlp_forward
#   - Normalizer, fit_normalizer
#   - TX_gen (flattened (t,x,y) points from surrogate simulation)
#   - key_main (PRNG key)
#   - cfg (for dtype / consistency)

from dataclasses import dataclass
import jax
import jax.numpy as jnp

# ----------------- Config for generator networks ------------------------------

@dataclass
class GenConfig:
    n_generators: int = 3
    hidden_tau: int = 32
    hidden_xi: int = 64
    hidden_beta: int = 64
    activation: str = "tanh"

gen_cfg = GenConfig()

# ----------------- Normalizers for generator inputs ---------------------------

# TX_gen has shape (B, 3) with columns [t, x, y]
t_flat_gen = TX_gen[:, 0:1]                 # (B,1)
txy_norm_gen = fit_normalizer(TX_gen)       # for ξ(t,x,y) and β(t,x,y)
t_norm_gen   = fit_normalizer(t_flat_gen)   # for τ(t)

# Backward-friendly alias name (old code used tx_norm_gen)
tx_norm_gen = txy_norm_gen

print("Gen t_norm mean/std:", t_norm_gen.mean, t_norm_gen.std)
print("Gen txy_norm mean/std:", txy_norm_gen.mean, txy_norm_gen.std)

# ----------------- Per-generator network builders -----------------------------

def tau_forward(params, t_norm, activation="tanh"):
    """
    Forward pass for τ(t).
    t_norm: (..., 1) normalized time.
    Returns (..., 1).
    """
    return mlp_forward(params, t_norm, activation=activation)[..., 0:1]

def xi_forward(params, txy_norm, activation="tanh"):
    """
    Forward pass for ξ(t,x,y) (2D state vector field component).
    txy_norm: (..., 3) normalized (t,x,y).
    Returns (..., 2) giving [ξ_x, ξ_y].
    """
    return mlp_forward(params, txy_norm, activation=activation)[..., 0:2]

def beta_forward(params, txy_norm, activation="tanh"):
    """
    Forward pass for β(t,x,y) (for FP / density pushforward form φ = β u).
    txy_norm: (..., 3) normalized (t,x,y).
    Returns (..., 1).
    """
    return mlp_forward(params, txy_norm, activation=activation)[..., 0:1]

def init_generator_params(key, gen_cfg: GenConfig):
    """
    Initialize parameters for all generators.
    Returns:
      params_gen = {
        "tau":  [params_tau_i  for i in range(m)],
        "xi":   [params_xi_i   for i in range(m)],   # outputs 2 dims
        "beta": [params_beta_i for i in range(m)],
      }
    """
    m = gen_cfg.n_generators
    keys = jax.random.split(key, 3 * m)

    params_tau_list  = []
    params_xi_list   = []
    params_beta_list = []

    for i in range(m):
        k_tau  = keys[3 * i]
        k_xi   = keys[3 * i + 1]
        k_beta = keys[3 * i + 2]

        # τ_i(t): input dim 1 -> hidden -> hidden -> output dim 1
        params_tau = init_mlp_params(
            k_tau,
            sizes=[1, gen_cfg.hidden_tau, gen_cfg.hidden_tau, 1],
        )

        # ξ_i(t,x,y): input dim 3 -> hidden -> hidden -> output dim 2
        params_xi = init_mlp_params(
            k_xi,
            sizes=[3, gen_cfg.hidden_xi, gen_cfg.hidden_xi, 2],
        )

        # β_i(t,x,y): input dim 3 -> hidden -> hidden -> output dim 1
        params_beta = init_mlp_params(
            k_beta,
            sizes=[3, gen_cfg.hidden_beta, gen_cfg.hidden_beta, 1],
        )

        params_tau_list.append(params_tau)
        params_xi_list.append(params_xi)
        params_beta_list.append(params_beta)

    return {
        "tau": params_tau_list,
        "xi": params_xi_list,
        "beta": params_beta_list,
    }

# Initialize generator parameters
key_main, key_gen = jax.random.split(key_main)
params_gen = init_generator_params(key_gen, gen_cfg)

# ----------------- Convenience evaluator for all generators -------------------

def eval_generators(params_gen, t, x, y, u=None, activation=None, return_phi=False):
    """
    Evaluate all generators at (t,x,y).

    If return_phi=True, pass u (same shape as broadcast(t,x,y)),
    and we return phi = beta(t,x,y) * u.

    Returns:
      tau_vals : (m, ...)         τ_i(t)
      xi_vals  : (m, ..., 2)      ξ_i(t,x,y) with last dim [ξ_x, ξ_y]
      beta_vals: (m, ...)         β_i(t,x,y)
      phi_vals : (m, ...)         φ_i = β_i * u   (optional)
    """
    if activation is None:
        activation = gen_cfg.activation

    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)
    y_arr = jnp.asarray(y, dtype=jnp.float64)

    # Broadcast to common shape
    t_b, x_b, y_b = jnp.broadcast_arrays(t_arr, x_arr, y_arr)
    B_shape = t_b.shape

    # Flatten
    t_flat = t_b.reshape(-1, 1)
    x_flat = x_b.reshape(-1, 1)
    y_flat = y_b.reshape(-1, 1)
    txy_flat = jnp.concatenate([t_flat, x_flat, y_flat], axis=1)  # (B,3)

    # Normalize
    t_norm   = t_norm_gen(t_flat)         # (B,1)
    txy_norm = txy_norm_gen(txy_flat)     # (B,3)

    tau_list, xi_list, beta_list = [], [], []

    for params_tau, params_xi, params_beta in zip(
        params_gen["tau"], params_gen["xi"], params_gen["beta"]
    ):
        tau_flat  = tau_forward(params_tau, t_norm, activation=activation)            # (B,1)
        xi_flat   = xi_forward(params_xi,  txy_norm, activation=activation)           # (B,2)
        beta_flat = beta_forward(params_beta, txy_norm, activation=activation)        # (B,1)

        tau_list.append(tau_flat.reshape(B_shape))                                    # (...)
        xi_list.append(xi_flat.reshape(*B_shape, 2))                                  # (...,2)
        beta_list.append(beta_flat.reshape(B_shape))                                  # (...)

    tau_vals  = jnp.stack(tau_list, axis=0)   # (m, ...)
    xi_vals   = jnp.stack(xi_list, axis=0)    # (m, ..., 2)
    beta_vals = jnp.stack(beta_list, axis=0)  # (m, ...)

    if return_phi:
        if u is None:
            raise ValueError("return_phi=True requires u to be provided.")
        u_arr = jnp.asarray(u, dtype=jnp.float64)
        u_b = jnp.broadcast_to(u_arr, B_shape)        # (...)
        phi_vals = beta_vals * u_b[None, ...]         # (m, ...)
        return tau_vals, xi_vals, beta_vals, phi_vals

    return tau_vals, xi_vals, beta_vals

# JIT-ed version for speed if desired
eval_generators_jit = jax.jit(
    eval_generators,
    static_argnames=("activation", "return_phi")
)

print(f"Initialized 2D generator nets with m = {gen_cfg.n_generators} generators.")

# Export to globals for later stages
globals().update({
    "gen_cfg": gen_cfg,
    "params_gen": params_gen,
    "t_norm_gen": t_norm_gen,
    "txy_norm_gen": txy_norm_gen,
    "tx_norm_gen": tx_norm_gen,      # alias
    "tau_forward": tau_forward,
    "xi_forward": xi_forward,
    "beta_forward": beta_forward,
    "eval_generators": eval_generators,
    "eval_generators_jit": eval_generators_jit,
})


In [ ]:
import jax
import jax.numpy as jnp

def eval_generators_tau_xi(params_gen, t, x, y, *, activation="tanh", normalize_txy=None):
    """
    2D SDE generator evaluator: returns only (tau, xi).

    Inputs:
      t, x, y: (B,) arrays
    Returns:
      tau_all: (m, B)
      xi_all:  (m, B, 2)   where last dim is [xi_x, xi_y]
    """
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.asarray(y, dtype=jnp.float64)

    if t.ndim != 1 or x.ndim != 1 or y.ndim != 1:
        raise ValueError("eval_generators_tau_xi expects t, x, y as 1D arrays of shape (B,)")

    # Optional preprocessing in raw (t,x,y) space
    if normalize_txy is None:
        t_raw, x_raw, y_raw = t, x, y
    else:
        t_raw, x_raw, y_raw = normalize_txy(t, x, y)
        t_raw = jnp.asarray(t_raw, dtype=jnp.float64)
        x_raw = jnp.asarray(x_raw, dtype=jnp.float64)
        y_raw = jnp.asarray(y_raw, dtype=jnp.float64)

    # Build (B,1) and (B,3) inputs
    t_col  = t_raw.reshape(-1, 1)                                # (B,1)
    x_col  = x_raw.reshape(-1, 1)                                # (B,1)
    y_col  = y_raw.reshape(-1, 1)                                # (B,1)
    txy_col = jnp.concatenate([t_col, x_col, y_col], axis=1)      # (B,3)

    # Apply the SAME normalizers used in generator training
    t_norm   = t_norm_gen(t_col)          # (B,1)
    txy_norm = txy_norm_gen(txy_col)      # (B,3)

    taus = []
    xis  = []

    for params_tau, params_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau_flat = tau_forward(params_tau, t_norm, activation=activation).reshape(-1)  # (B,)
        xi_flat  = xi_forward(params_xi,  txy_norm, activation=activation)             # (B,2)
        taus.append(tau_flat)
        xis.append(xi_flat)

    tau_all = jnp.stack(taus, axis=0)  # (m,B)
    xi_all  = jnp.stack(xis,  axis=0)  # (m,B,2)
    return tau_all, xi_all

# JIT wrapper (recommended)
eval_generators_tau_xi_jit = jax.jit(
    eval_generators_tau_xi,
    static_argnames=("activation", "normalize_txy"),
)


# Algebraic Losses

## Loss 1

In [ ]:
# ============================ S1 — Lie bracket closure + constancy (2D version) ============================
# Compatible with:
#   - params_gen: {"tau": [params_tau_i], "xi": [params_xi_i]}
#   - gen_cfg.n_generators
#   - t_norm_gen, txy_norm_gen
#   - tau_forward, xi_forward
#
# Vector fields: X_i = τ_i(t) ∂_t + ξ_i^x(t,x,y) ∂_x + ξ_i^y(t,x,y) ∂_y.
# We enforce:
#   - [X_i, X_j] stays in span{X_k} via projection in R^3 at each (t,x,y)
#   - structure coefficients c_ij^k approximately constant over (t,x,y).

import jax
import jax.numpy as jnp

def _ordered_pair_indices(n: int):
    """
    Return (i,j) pairs with i != j, in a deterministic order.
    Used to enumerate all brackets [X_i, X_j].
    """
    idx = jnp.arange(n, dtype=jnp.int32)
    ii  = jnp.repeat(idx, repeats=n - 1)
    base = jnp.arange(n - 1, dtype=jnp.int32)
    i_col = idx[:, None]
    jj_mat = base + (base >= i_col).astype(jnp.int32)
    jj = jj_mat.reshape(-1)
    return ii, jj

def make_s1_lie_loss(n_generators: int, rcond: float = 1e-6):
    """
    S1 Lie algebra loss (2D spatial):
      - Closure: project [X_i, X_j] onto span{X_k} at each (t,x,y) and penalize residual.
      - Constancy: penalize variation of structure coefficients c_ij^k over (t,x,y).

    Args:
      n_generators: number of learned generators m.
      rcond: small regularization parameter for Gram matrix inversion.

    Returns:
      loss_fn(params_gen, txy_batch) -> (scalar_loss, aux_dict)
    """
    idx_i, idx_j = _ordered_pair_indices(n_generators)
    K = int(idx_i.shape[0])
    reg = jnp.asarray(rcond, dtype=jnp.float64) ** 2

    # ---------- helper: τ_i(t) and ∂_t τ_i(t) ----------------------

    def _tau_val_and_dt(params_tau_i, t_scalar):
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)    # (1,1)
            t_norm = t_norm_gen(t_arr)                        # (1,1)
            out = tau_forward(params_tau_i, t_norm, activation=gen_cfg.activation)
            return out[0, 0]  # scalar
        tau_val = tau_scalar(t_scalar)
        tau_t = jax.grad(tau_scalar)(t_scalar)
        return tau_val, tau_t

    # ---------- helper: ξ_i(t,x,y) and partials ---------------------

    def _xi_val_and_derivs(params_xi_i, t_scalar, x_scalar, y_scalar):
        """
        ξ_i: (2,) = [ξ_i^x, ξ_i^y]
        Returns:
          xi_val : (2,)
          xi_t   : (2,)  ∂_t ξ
          xi_x   : (2,)  ∂_x ξ
          xi_y   : (2,)  ∂_y ξ
        """
        def xi_fun(tt, xx, yy):
            txy_arr = jnp.asarray([[tt, xx, yy]], dtype=jnp.float64)  # (1,3)
            txy_norm = txy_norm_gen(txy_arr)
            out = xi_forward(params_xi_i, txy_norm, activation=gen_cfg.activation)
            # out expected shape (1,2) or (1,2-ish); take first row
            return out[0]  # (2,)

        xi_val = xi_fun(t_scalar, x_scalar, y_scalar)
        xi_t = jax.jacfwd(lambda tt: xi_fun(tt, x_scalar, y_scalar))(t_scalar)  # (2,)
        xi_x = jax.jacfwd(lambda xx: xi_fun(t_scalar, xx, y_scalar))(x_scalar)  # (2,)
        xi_y = jax.jacfwd(lambda yy: xi_fun(t_scalar, x_scalar, yy))(y_scalar)  # (2,)
        return xi_val, xi_t, xi_x, xi_y

    # ---------- helper: fields and derivatives at a single (t,x,y) -------------

    def _fields_and_derivs_at_point(params_gen, t_scalar, x_scalar, y_scalar):
        """
        Compute τ_i, ξ_i and derivatives at (t,x,y) for all generators.
        Returns:
          tau   : (m,)
          xi    : (m,2)
          tau_t : (m,)
          xi_t  : (m,2)
          xi_x  : (m,2)
          xi_y  : (m,2)
        """
        tau_params = params_gen["tau"]
        xi_params  = params_gen["xi"]

        tau_list, tau_t_list = [], []
        xi_list, xi_t_list, xi_x_list, xi_y_list = [], [], [], []

        for p_tau, p_xi in zip(tau_params, xi_params):
            tau_i, tau_t_i = _tau_val_and_dt(p_tau, t_scalar)
            xi_i, xi_t_i, xi_x_i, xi_y_i = _xi_val_and_derivs(p_xi, t_scalar, x_scalar, y_scalar)

            tau_list.append(tau_i)
            tau_t_list.append(tau_t_i)
            xi_list.append(xi_i)
            xi_t_list.append(xi_t_i)
            xi_x_list.append(xi_x_i)
            xi_y_list.append(xi_y_i)

        tau   = jnp.stack(tau_list, axis=0)       # (m,)
        tau_t = jnp.stack(tau_t_list, axis=0)     # (m,)
        xi    = jnp.stack(xi_list, axis=0)        # (m,2)
        xi_t  = jnp.stack(xi_t_list, axis=0)      # (m,2)
        xi_x  = jnp.stack(xi_x_list, axis=0)      # (m,2)
        xi_y  = jnp.stack(xi_y_list, axis=0)      # (m,2)

        return tau, xi, tau_t, xi_t, xi_x, xi_y

    # ---------- helper: bracket closure + coefficients at a point ------------

    def _point_err_and_C(tau, xi, tau_t, xi_t, xi_x, xi_y):
        """
        Single-point closure + structure coefficients.

        X_i = τ_i ∂_t + ξ_i^x ∂_x + ξ_i^y ∂_y

        Bracket components:
          [X_i, X_j]^t = τ_i τ_{j,t} - τ_j τ_{i,t}

          [X_i, X_j]^x = τ_i ξ^x_{j,t} + ξ_i^x ξ^x_{j,x} + ξ_i^y ξ^x_{j,y}
                         - (τ_j ξ^x_{i,t} + ξ_j^x ξ^x_{i,x} + ξ_j^y ξ^x_{i,y})

          [X_i, X_j]^y = τ_i ξ^y_{j,t} + ξ_i^x ξ^y_{j,x} + ξ_i^y ξ^y_{j,y}
                         - (τ_j ξ^y_{i,t} + ξ_j^x ξ^y_{i,x} + ξ_j^y ξ^y_{i,y})
        """
        # V: (3,m) with rows [τ; ξ^x; ξ^y]
        V = jnp.stack([tau, xi[:, 0], xi[:, 1]], axis=0)  # (3,m)

        # Slice i,j components
        tau_i, tau_j = tau[idx_i], tau[idx_j]           # (K,)
        tau_t_i, tau_t_j = tau_t[idx_i], tau_t[idx_j]   # (K,)

        xi_i_x, xi_i_y = xi[idx_i, 0], xi[idx_i, 1]     # (K,), (K,)
        xi_j_x, xi_j_y = xi[idx_j, 0], xi[idx_j, 1]

        xi_t_i_x, xi_t_i_y = xi_t[idx_i, 0], xi_t[idx_i, 1]
        xi_t_j_x, xi_t_j_y = xi_t[idx_j, 0], xi_t[idx_j, 1]

        xi_x_i_x, xi_x_i_y = xi_x[idx_i, 0], xi_x[idx_i, 1]
        xi_x_j_x, xi_x_j_y = xi_x[idx_j, 0], xi_x[idx_j, 1]

        xi_y_i_x, xi_y_i_y = xi_y[idx_i, 0], xi_y[idx_i, 1]
        xi_y_j_x, xi_y_j_y = xi_y[idx_j, 0], xi_y[idx_j, 1]

        # Bracket t-component
        a = tau_i * tau_t_j - tau_j * tau_t_i

        # Bracket x-component
        b = (
            tau_i * xi_t_j_x + xi_i_x * xi_x_j_x + xi_i_y * xi_y_j_x
            - (tau_j * xi_t_i_x + xi_j_x * xi_x_i_x + xi_j_y * xi_y_i_x)
        )

        # Bracket y-component
        c = (
            tau_i * xi_t_j_y + xi_i_x * xi_x_j_y + xi_i_y * xi_y_j_y
            - (tau_j * xi_t_i_y + xi_j_x * xi_x_i_y + xi_j_y * xi_y_i_y)
        )

        B = jnp.stack([a, b, c], axis=0)  # (3,K)

        # Project B onto span(V) in R^3
        G = V @ V.T                                    # (3,3)
        G_reg = G + reg * jnp.eye(3, dtype=G.dtype)
        X = jnp.linalg.solve(G_reg, B)                 # (3,K)
        C = V.T @ X                                    # (m,K)
        P_B = V @ C                                    # (3,K)
        E = B - P_B                                    # (3,K)

        err = jnp.sum(jnp.abs(E))
        return err, C

    # ---------- main loss over batch -----------------------------------------

    def _loss_impl(params_gen, txy_batch: jnp.ndarray):
        """
        txy_batch: (B,3) with columns [t, x, y].
        """
        def eval_at_z(z):
            t_z, x_z, y_z = z[0], z[1], z[2]
            return _fields_and_derivs_at_point(params_gen, t_z, x_z, y_z)

        taus, xis, tau_ts, xi_ts, xi_xs, xi_ys = jax.vmap(eval_at_z)(txy_batch)
        # taus:   (B,m)
        # xis:    (B,m,2)
        # tau_ts: (B,m)
        # xi_ts, xi_xs, xi_ys: (B,m,2)

        errs, Cs = jax.vmap(_point_err_and_C)(taus, xis, tau_ts, xi_ts, xi_xs, xi_ys)
        # errs: (B,)
        # Cs:   (B, m, K)

        error_sum = jnp.sum(errs)

        # Constancy across batch: variance of C over (t,x,y)
        C_var = jnp.var(Cs, axis=0)  # (m, K)
        var_sum = jnp.sum(C_var)

        total = error_sum + var_sum
        aux = {"error_sum": error_sum, "var_sum": var_sum}
        return total, aux

    return jax.jit(_loss_impl)


## Loss 2 - Jacobi Identity

In [ ]:
# ============================ S2 — Jacobi identity (nested brackets, 2D SDE version) ============================
# Uses per-generator Xi_i(t,x,y) with its own (3x3) Jacobian and (3x3x3) Hessian,
# so that all matrix–vector products are dimensionally consistent.
#
# Vector fields live on (t,x,y):  X_i = τ_i(t) ∂_t + ξ_i^x(t,x,y) ∂_x + ξ_i^y(t,x,y) ∂_y.

import jax
import jax.numpy as jnp

def make_s2_jacobi_loss_nested(n_generators: int):
    # All distinct index triples i < j < k
    triples = [
        (i, j, k)
        for i in range(n_generators)
        for j in range(i + 1, n_generators)
        for k in range(j + 1, n_generators)
    ]
    if not triples:
        def _zero(params_gen, txy_batch):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64),
                "num_triples": 0,
            }
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    # All 6 permutations for symmetrized Jacobi expression
    perms6 = jnp.array(
        [[0, 1, 2],
         [0, 2, 1],
         [1, 0, 2],
         [1, 2, 0],
         [2, 0, 1],
         [2, 1, 0]],
        dtype=jnp.int32,
    )

    # -------- helper: F, J, H at a single (t,x,y) via per-generator Xi_i -------

    def _fields_jac_hess(params_gen, z):
        """
        Compute:
          F: (m,3)      vector field values at z = (t,x,y)
          J: (m,3,3)    Jacobians D X_i(z)
          H: (m,3,3,3)  Hessians  D^2 X_i(z)
        for all generators X_i.
        """
        t_z, x_z, y_z = z[0], z[1], z[2]

        F_list = []
        J_list = []
        H_list = []

        # Iterate over generators; loop size m is static so jit-friendly.
        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx, yy = zz[0], zz[1], zz[2]

                t_arr   = jnp.asarray([[tt]], dtype=jnp.float64)            # (1,1)
                txy_arr = jnp.asarray([[tt, xx, yy]], dtype=jnp.float64)    # (1,3)

                t_norm   = t_norm_gen(t_arr)        # (1,1)
                txy_norm = txy_norm_gen(txy_arr)    # (1,3)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]  # scalar

                xi_xy = xi_forward(
                    params_xi_i,
                    txy_norm,
                    activation=gen_cfg.activation,
                )[0]     # (2,) = [xi^x, xi^y]

                return jnp.array([tau_val, xi_xy[0], xi_xy[1]], dtype=jnp.float64)  # (3,)

            # Value, Jacobian, Hessian for generator i
            Fi = Xi(z)                                                # (3,)
            Ji = jax.jacobian(Xi)(z)                                  # (3,3)
            Hi = jax.jacobian(lambda zz: jax.jacobian(Xi)(zz))(z)     # (3,3,3)

            F_list.append(Fi)
            J_list.append(Ji)
            H_list.append(Hi)

        F = jnp.stack(F_list, axis=0)   # (m,3)
        J = jnp.stack(J_list, axis=0)   # (m,3,3)
        H = jnp.stack(H_list, axis=0)   # (m,3,3,3)

        return F, J, H

    # ------------- helper: basic bracket and nested bracket algebra ----------

    def _bracket_val(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        where F_i ∈ R^3, J_i ∈ R^{3x3}.
        Returns a 3-vector (in (∂_t, ∂_x, ∂_y) basis).
        """
        Jp, Jq = J[p], J[q]   # (3,3)
        fp, fq = F[p], F[q]   # (3,)
        return (Jq @ fp) - (Jp @ fq)   # (3,)

    def _dir_along(F, J, H, r, p, q):
        """
        Directional action of [X_p, X_q] on X_r, using first and second derivatives,
        generalized to 3D coordinate (t,x,y).
        """
        Jr, Jp, Jq = J[r], J[p], J[q]     # (3,3)
        Hp, Hq = H[p], H[q]               # (3,3,3)
        fr, fp, fq = F[r], F[p], F[q]     # (3,)

        # (3,) results
        t1 = Jq @ (Jp @ fr)
        t2 = ((Hq * fr[None, None, :]).sum(axis=2)) @ fp   # (3,3) @ (3,) -> (3,)
        t3 = Jp @ (Jq @ fr)
        t4 = ((Hp * fr[None, None, :]).sum(axis=2)) @ fq

        return t1 + t2 - t3 - t4

    def _double_bracket(F, J, H, r, p, q):
        """
        Nested bracket [[X_r, X_p], X_q] at a point.
        """
        inner = _bracket_val(F, J, p, q)      # (3,)
        return _dir_along(F, J, H, r, p, q) - (J[r] @ inner)

    def _jacobi_one_order(F, J, H, u, v, w):
        """
        Jacobi combination for one ordering (u, v, w):
          [[X_u, X_v], X_w] + [[X_w, X_u], X_v] + [[X_v, X_w], X_u]
        """
        return (
            _double_bracket(F, J, H, u, v, w)
            + _double_bracket(F, J, H, w, u, v)
            + _double_bracket(F, J, H, v, w, u)
        )

    def _triple_sum_over_6(F, J, H, i, j, k):
        """
        Symmetrize over all 6 permutations of (i,j,k), summing absolute values.
        """
        inds = jnp.array([i, j, k], dtype=jnp.int32)

        def _one_perm(p):
            u, v, w = inds[p[0]], inds[p[1]], inds[p[2]]
            r = _jacobi_one_order(F, J, H, u, v, w)  # (3,)
            return jnp.sum(jnp.abs(r))

        vals = jax.vmap(_one_perm)(perms6)  # (6,)
        return jnp.sum(vals)

    def _point_loss(params_gen, z):
        """
        Jacobi loss at a single point z = (t,x,y), summed over all triples (i,j,k).
        """
        F, J, H = _fields_jac_hess(params_gen, z)
        per_tr = jax.vmap(
            lambda a, b, c: _triple_sum_over_6(F, J, H, a, b, c)
        )(tri_i, tri_j, tri_k)  # (num_triples,)
        return jnp.sum(per_tr)

    _point_loss_jit = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch: jnp.ndarray):
        """
        txy_batch: (B,3) array with columns [t, x, y].
        Returns:
          total_loss, {
              "per_point": (B,),
              "num_triples": int,
          }
        """
        per_point = jax.vmap(lambda z: _point_loss_jit(params_gen, z))(txy_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_triples": int(tri_i.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 3 - Skewsymmetry

In [ ]:
# ============================ S3 — Skew-symmetry (2D SDE version) ============================
# Uses per-generator Xi_i(t,x,y) so that:
#   F: (m,3),  J: (m,3,3)
# and [X_i, X_j](z) = J_j @ F_i - J_i @ F_j is always well-typed.
#
# Vector fields on (t,x,y):
#   X_i = τ_i(t) ∂_t + ξ_i^x(t,x,y) ∂_x + ξ_i^y(t,x,y) ∂_y

import jax
import jax.numpy as jnp

def make_s3_skewsym_loss(n_generators: int):
    # All distinct pairs i < j
    pairs = [(i, j) for i in range(n_generators) for j in range(i + 1, n_generators)]
    if not pairs:
        def _zero(params_gen, txy_batch):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64),
                "num_pairs": 0,
            }
        return jax.jit(_zero)

    pi = jnp.array([p[0] for p in pairs], dtype=jnp.int32)
    pj = jnp.array([p[1] for p in pairs], dtype=jnp.int32)

    # ---------- helper: F and J at a single (t,x,y) --------------------------

    def _fields_and_jac(params_gen, z):
        """
        Compute:
          F: (m,3)    vector field values at z = (t,x,y)
          J: (m,3,3)  Jacobians w.r.t (t,x,y)
        for all generators X_i.
        """
        t_z, x_z, y_z = z[0], z[1], z[2]

        F_list = []
        J_list = []

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx, yy = zz[0], zz[1], zz[2]

                t_arr   = jnp.asarray([[tt]], dtype=jnp.float64)            # (1,1)
                txy_arr = jnp.asarray([[tt, xx, yy]], dtype=jnp.float64)    # (1,3)

                t_norm   = t_norm_gen(t_arr)         # (1,1)
                txy_norm = txy_norm_gen(txy_arr)     # (1,3)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]  # scalar

                xi_xy = xi_forward(
                    params_xi_i,
                    txy_norm,
                    activation=gen_cfg.activation,
                )[0]     # (2,) = [xi^x, xi^y]

                return jnp.array([tau_val, xi_xy[0], xi_xy[1]], dtype=jnp.float64)  # (3,)

            Fi = Xi(z)                 # (3,)
            Ji = jax.jacobian(Xi)(z)   # (3,3)

            F_list.append(Fi)
            J_list.append(Ji)

        F = jnp.stack(F_list, axis=0)  # (m,3)
        J = jnp.stack(J_list, axis=0)  # (m,3,3)
        return F, J

    # ---------- bracket at a point ------------------------------------------

    def _bracket_val(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        """
        Jp, Jq = J[p], J[q]   # (3,3)
        fp, fq = F[p], F[q]   # (3,)
        return (Jq @ fp) - (Jp @ fq)   # (3,)

    def _point_loss(params_gen, z):
        """
        Skew-symmetry loss at a single point z = (t,x,y):
          sum_{i<j} || [X_i, X_j] + [X_j, X_i] ||_1
        """
        F, J = _fields_and_jac(params_gen, z)

        def one(i, j):
            r = _bracket_val(F, J, i, j) + _bracket_val(F, J, j, i)
            return jnp.sum(jnp.abs(r))

        vals = jax.vmap(one)(pi, pj)  # (num_pairs,)
        return jnp.sum(vals)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch: jnp.ndarray):
        """
        txy_batch: (B,3) with columns [t, x, y].
        Returns:
          total_loss, {
              "per_point": (B,),
              "num_pairs": int,
          }
        """
        per_point = jax.vmap(lambda z: _pl(params_gen, z))(txy_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_pairs": int(pi.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 4 - Bilinearity

In [ ]:
# ============================ S4 — Bilinearity (2D SDE version) ============================
# Uses per-generator Xi_i(t,x,y) so F and J are well-typed:
#   F: (m,3), J: (m,3,3)
# Checks:
#   [c u + c' v, w] = c [u, w] + c' [v, w]
#   [u, c v + c' w] = c [u, v] + c' [u, w]
#
# Vector fields on (t,x,y):
#   X_i = τ_i(t) ∂_t + ξ_i^x(t,x,y) ∂_x + ξ_i^y(t,x,y) ∂_y

import jax
import jax.numpy as jnp

def make_s4_bilinearity_loss(
    n_generators: int,
    num_cc: int = 4,
    cc_list=None,
    normalize: bool = True,
):
    # All distinct triples i < j < k
    triples = [
        (i, j, k)
        for i in range(n_generators)
        for j in range(i + 1, n_generators)
        for k in range(j + 1, n_generators)
    ]
    if not triples:
        def _zero(params_gen, txy_batch, key=None):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64),
            }
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    perms6 = jnp.array(
        [[0, 1, 2],
         [0, 2, 1],
         [1, 0, 2],
         [1, 2, 0],
         [2, 0, 1],
         [2, 1, 0]],
        dtype=jnp.int32,
    )

    # ----------- helper: F and J at a single (t,x,y) -------------------------

    def _fields_and_jac(params_gen, z):
        """
        Compute:
          F: (m,3)   vector field values at z = (t,x,y)
          J: (m,3,3) Jacobians w.r.t (t,x,y)
        for all generators X_i.
        """
        t_z, x_z, y_z = z[0], z[1], z[2]

        F_list = []
        J_list = []

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx, yy = zz[0], zz[1], zz[2]

                t_arr   = jnp.asarray([[tt]], dtype=jnp.float64)              # (1,1)
                txy_arr = jnp.asarray([[tt, xx, yy]], dtype=jnp.float64)      # (1,3)

                t_norm   = t_norm_gen(t_arr)          # (1,1)
                txy_norm = txy_norm_gen(txy_arr)      # (1,3)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]  # scalar

                xi_xy = xi_forward(
                    params_xi_i,
                    txy_norm,
                    activation=gen_cfg.activation,
                )[0]     # (2,) = [xi^x, xi^y]

                return jnp.array([tau_val, xi_xy[0], xi_xy[1]], dtype=jnp.float64)  # (3,)

            Fi = Xi(z)                # (3,)
            Ji = jax.jacobian(Xi)(z)  # (3,3)

            F_list.append(Fi)
            J_list.append(Ji)

        F = jnp.stack(F_list, axis=0)  # (m,3)
        J = jnp.stack(J_list, axis=0)  # (m,3,3)
        return F, J

    # ----------- bracket and bilinearity terms at a point --------------------

    def _bracket(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        """
        Jp, Jq = J[p], J[q]   # (3,3)
        fp, fq = F[p], F[q]   # (3,)
        return (Jq @ fp) - (Jp @ fq)   # (3,)

    def _triple_terms(F, J, i, j, k, cc):
        """
        Bilinearity residuals for a single triple (i,j,k) at a fixed point,
        averaged over coefficient pairs in cc.
        """
        inds = jnp.array([i, j, k], dtype=jnp.int32)

        def one_perm(p):
            u, v, w = inds[p[0]], inds[p[1]], inds[p[2]]
            fu, fv, fw = F[u], F[v], F[w]        # (3,)
            Ju, Jv, Jw = J[u], J[v], J[w]        # (3,3)

            def one_cc(cpair):
                c, cp = cpair[0], cpair[1]

                # Linear combinations in the first slot
                f_uv = c * fu + cp * fv
                J_uv = c * Ju + cp * Jv

                # Linear combinations in the second slot
                f_vw = c * fv + cp * fw
                J_vw = c * Jv + cp * Jw

                # [c u + c' v, w]
                term1 = (Jw @ f_uv) - (J_uv @ fw)
                rhs1  = c * _bracket(F, J, u, w) + cp * _bracket(F, J, v, w)
                r1 = term1 - rhs1

                # [u, c v + c' w]
                term2 = (J_vw @ fu) - (Ju @ f_vw)
                rhs2  = c * _bracket(F, J, u, v) + cp * _bracket(F, J, u, w)
                r2 = term2 - rhs2

                if normalize:
                    denom = jnp.abs(c) + jnp.abs(cp) + 1e-12
                    r1 = r1 / denom
                    r2 = r2 / denom

                return jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))

            vals_cc = jax.vmap(one_cc)(cc)  # (num_cc,)
            return jnp.mean(vals_cc)

        vals = jax.vmap(one_perm)(perms6)  # (6,)
        return jnp.sum(vals)

    def _point_loss(params_gen, z, cc):
        """
        Bilinearity loss at a single point z = (t,x,y), summed over all triples.
        """
        F, J = _fields_and_jac(params_gen, z)
        per_tr = jax.vmap(
            lambda a, b, c_: _triple_terms(F, J, a, b, c_, cc)
        )(tri_i, tri_j, tri_k)  # (num_triples,)
        return jnp.sum(per_tr)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch: jnp.ndarray, key=None):
        """
        txy_batch: (B,3) with columns [t, x, y].
        key: optional PRNGKey for sampling coefficient pairs if cc_list is None.
        """
        if cc_list is not None:
            cc = jnp.asarray(cc_list, dtype=jnp.float64)  # (num_cc, 2)
        else:
            key = jax.random.PRNGKey(0) if key is None else key
            cc = jax.random.uniform(
                key,
                (num_cc, 2),
                minval=-1.0,
                maxval=1.0,
                dtype=jnp.float64,
            )

        per_point = jax.vmap(lambda z: _pl(params_gen, z, cc))(txy_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_triples": int(tri_i.shape[0]),
            "num_cc": int(cc.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 5 - Functional Independence

In [ ]:
# ============================ S5 — Column independence (supports 1D or 2D xi; supports extra returns) ============================
# Works with:
#   - 1D case: eval_generators_jit(params_gen, t, x) -> (tau:(m,N), xi:(m,N), beta/...)
#   - 2D case: eval_generators_jit(params_gen, t, x, y) -> (tau:(m,N), xi:(m,N,2), beta/...)
#
# Builds A by stacking per-point components:
#   - 1D: rows are [tau_i(t_n), xi_i(t_n,x_n)]  -> A ∈ R^{2N x m}
#   - 2D: rows are [tau_i(t_n), xi^x_i(t_n,x_n,y_n), xi^y_i(t_n,x_n,y_n)] -> A ∈ R^{3N x m}

import jax
import jax.numpy as jnp

def make_s5_column_independence_loss(
    n_generators: int,
    *,
    mode: str = "sigma",
    tau: float = 0.0,          # margin (kept name for compatibility)
    eps: float = 1e-12,
):
    mode = "sigma" if mode == "sigma" else "corr_l2"
    mode_code = 0 if mode == "sigma" else 1
    tau_margin = tau

    def _A_from_batch(params_gen, txy_batch: jnp.ndarray):
        """
        Build A ∈ R^{dN x m} from a batch of points (d = 2 or 3):
          - rows: components of X_i stacked over points
          - cols: generators i = 0..m-1

        txy_batch:
          - (N,2) with columns [t, x]  (legacy)
          - (N,3) with columns [t, x, y]  (2D SDE)
        """
        N, D = txy_batch.shape
        t_batch = txy_batch[:, 0]  # (N,)
        x_batch = txy_batch[:, 1]  # (N,)

        if D == 2:
            outs = eval_generators_jit(params_gen, t_batch, x_batch)
        elif D == 3:
            y_batch = txy_batch[:, 2]
            outs = eval_generators_jit(params_gen, t_batch, x_batch, y_batch)
        else:
            raise ValueError(f"Expected txy_batch with 2 or 3 columns; got shape {txy_batch.shape}")

        tau_vals = outs[0]  # (m,N)
        xi_vals  = outs[1]  # (m,N) or (m,N,2)

        # Stack components per generator per point:
        #   1D: comp = (m,N,2)  with last axis [tau, xi]
        #   2D: comp = (m,N,3)  with last axis [tau, xi_x, xi_y]
        if xi_vals.ndim == 2:
            comp = jnp.stack([tau_vals, xi_vals], axis=2)  # (m,N,2)
        elif xi_vals.ndim == 3 and xi_vals.shape[-1] == 2:
            comp = jnp.concatenate([tau_vals[..., None], xi_vals], axis=2)  # (m,N,3)
        else:
            raise ValueError(f"Unexpected xi_vals shape {xi_vals.shape}; expected (m,N) or (m,N,2)")

        # (m,N,d) -> (N,d,m) -> (dN,m)
        comp_Bdm = jnp.transpose(comp, (1, 2, 0))           # (N,d,m)
        A = comp_Bdm.reshape(-1, n_generators)              # (dN,m)
        return A

    def _loss_impl(params_gen, txy_batch: jnp.ndarray):
        A = _A_from_batch(params_gen, txy_batch)  # (dN, m)

        # Normalize columns
        col_norms = jnp.linalg.norm(A, axis=0) + eps   # (m,)
        Ahat = A / col_norms                           # (dN, m)

        # Gram matrix of normalized columns
        G = Ahat.T @ Ahat                              # (m, m)

        if mode_code == 0:
            lam = jnp.linalg.eigvalsh(G)
            lam_min = jnp.clip(jnp.min(lam), 0.0, None)
            sigma_min = jnp.sqrt(lam_min)
            loss = jnp.maximum(
                0.0,
                jnp.asarray(tau_margin, dtype=G.dtype) - sigma_min,
            )
            aux = {"sigma_min": sigma_min}
        else:
            I = jnp.eye(G.shape[0], dtype=G.dtype)
            off = G - I
            off = off - jnp.diag(jnp.diag(off))
            loss = jnp.sum(off * off)
            aux = {"gram_diag_mean": jnp.mean(jnp.diag(G))}

        return loss, aux

    return jax.jit(_loss_impl)


# SDE Symmetry Losses

## Loss 6 - SDE Symmetry DE

In [ ]:
# ============================ S6 — 2D SDE determining-equation loss (Gaeta–Quintero style) ============================
# 2D projectable Ito SDE symmetry determining equations for:
#   dZ = f(t,Z) dt + σ(t,Z) dW
# with Z=(x,y) ∈ R^2, f ∈ R^2, σ ∈ R^{2×m}.
#
# For each generator X_i = τ_i(t) ∂_t + ξ_i(t,x,y)·∇, where ξ_i ∈ R^2:
#
#   r1 = ξ_t + (Jξ) f - (Jf) ξ - τ f_t - f τ_t + 0.5 * (a : Hess(ξ)) = 0    (vector in R^2)
#   r2 = (Jξ) σ - (ξ·∇)σ - τ σ_t - 0.5 σ τ_t = 0                           (matrix in R^{2×m})
#
# where a = σ σ^T and (a : Hess(ξ))_p = Σ_{j,k} a_{jk} ∂_{jk} ξ_p.

import jax
import jax.numpy as jnp

def make_s6_commutator_loss_ito_2d(*, mu_fn, sig_fn, use_abs: bool = False):
    """
    Args:
      mu_fn(t, x, y)   -> (2,) drift f
      sig_fn(t, x, y)  -> (2,) diagonal entries [σx, σy]  OR  (2,2) full σ-matrix
      use_abs: if False uses L2 (squared); if True uses L1 (abs)

    Returns:
      loss_fn(params_gen, txy_batch) -> (loss_scalar, aux_dict)
        - params_gen: {"tau":[...], "xi":[...]} where xi_forward returns (..,2)
        - txy_batch:  (B,3) with columns [t, x, y]
        - aux_dict:   {"per_point": (B,)}
    """

    # pick the joint normalizer name flexibly (likely rename tx_norm_gen -> txy_norm_gen in 2D)
    def _txynorm(arr):
        if "txy_norm_gen" in globals():
            return globals()["txy_norm_gen"](arr)
        return tx_norm_gen(arr)

    # ----- τ(t) and τ_t(t) ---------------------------------------------------
    def tau_val_and_dt(params_tau, t_scalar):
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)  # (1,1)
            t_norm = t_norm_gen(t_arr)
            out = tau_forward(params_tau, t_norm, activation=gen_cfg.activation)
            return out[0, 0]
        tau_val = tau_scalar(t_scalar)
        tau_t = jax.grad(tau_scalar)(t_scalar)
        return tau_val, tau_t

    # ----- ξ(t,x,y) and derivatives ------------------------------------------
    def xi_val_and_derivs(params_xi, t_scalar, x_scalar, y_scalar):
        """
        Returns:
          xi      : (2,)
          xi_t    : (2,)
          Jxi     : (2,2)   wrt (x,y)
          Hxi     : (2,2,2) Hessians wrt (x,y), with Hxi[p,j,k] = ∂_{jk} ξ_p
        """
        z = jnp.array([x_scalar, y_scalar], dtype=jnp.float64)

        def xi_vec(tt, zz):
            # Build (1,3) input [t,x,y] -> normalize -> xi_forward -> (2,)
            txy = jnp.asarray([[tt, zz[0], zz[1]]], dtype=jnp.float64)  # (1,3)
            txy_n = _txynorm(txy)
            out = xi_forward(params_xi, txy_n, activation=gen_cfg.activation)
            out = out.reshape(-1)
            if out.shape[0] != 2:
                raise ValueError(f"xi_forward must output 2 components in 2D; got shape {out.shape}")
            return out  # (2,)

        # Values
        xi_val = xi_vec(t_scalar, z)  # (2,)

        # Time derivative (2,)
        xi_t = jax.jacobian(lambda tt: xi_vec(tt, z))(t_scalar)

        # Jacobian wrt (x,y): (2,2)
        Jxi = jax.jacobian(lambda zz: xi_vec(t_scalar, zz))(z)

        # Hessian wrt (x,y): (2,2,2)
        Hxi = jax.jacobian(lambda zz: jax.jacobian(lambda z2: xi_vec(t_scalar, z2))(zz))(z)

        return xi_val, xi_t, Jxi, Hxi

    # ----- f, σ and derivatives ----------------------------------------------
    def f_sigma_and_derivs(t_scalar, x_scalar, y_scalar):
        z = jnp.array([x_scalar, y_scalar], dtype=jnp.float64)

        def f_of_t(tt):
            return mu_fn(tt, x_scalar, y_scalar)  # (2,)

        def f_of_z(zz):
            return mu_fn(t_scalar, zz[0], zz[1])  # (2,)

        f_val = mu_fn(t_scalar, x_scalar, y_scalar)                 # (2,)
        f_t   = jax.jacobian(f_of_t)(t_scalar)                      # (2,)
        Jf    = jax.jacobian(f_of_z)(z)                             # (2,2)

        def sigma_raw(tt, zz):
            return sig_fn(tt, zz[0], zz[1])

        sig_val_raw = sigma_raw(t_scalar, z)
        if sig_val_raw.ndim == 1 and sig_val_raw.shape[0] == 2:
            # interpret as diagonal entries => σ ∈ R^{2×2} diagonal
            sigma_mat = jnp.diag(sig_val_raw)                       # (2,2)
        elif sig_val_raw.shape == (2, 2):
            sigma_mat = sig_val_raw
        else:
            raise ValueError(f"sig_fn must return shape (2,) or (2,2); got {sig_val_raw.shape}")

        # σ_t: (2,2)
        sigma_t = jax.jacobian(lambda tt: (
            jnp.diag(sigma_raw(tt, z)) if (sigma_raw(tt, z).ndim == 1) else sigma_raw(tt, z)
        ))(t_scalar)

        # Dσ wrt z: (2,2,2)  (i,j,k) = ∂_{z_k} σ_{ij}
        def sigma_mat_of_z(zz):
            s = sigma_raw(t_scalar, zz)
            return jnp.diag(s) if (s.ndim == 1) else s

        Dsigma = jax.jacobian(sigma_mat_of_z)(z)                    # (2,2,2)

        return f_val, f_t, Jf, sigma_mat, sigma_t, Dsigma

    # ----- per-point residual sum over generators ----------------------------
    def _point_residual(params_gen, z_txy):
        t, x, y = z_txy[0], z_txy[1], z_txy[2]

        f_val, f_t, Jf, sigma_mat, sigma_t, Dsigma = f_sigma_and_derivs(t, x, y)
        a = sigma_mat @ sigma_mat.T  # (2,2)

        total = jnp.array(0.0, dtype=jnp.float64)

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):
            tau_i, tau_t_i = tau_val_and_dt(params_tau_i, t)
            xi_i, xi_t_i, Jxi_i, Hxi_i = xi_val_and_derivs(params_xi_i, t, x, y)

            # diffusion contraction: (2,) where component p is Σ_{j,k} a_{jk} Hxi[p,j,k]
            diff_term = jnp.einsum("jk,pjk->p", a, Hxi_i)

            # r1 ∈ R^2
            r1 = (
                xi_t_i
                + (Jxi_i @ f_val)
                - (Jf @ xi_i)
                - tau_i * f_t
                - f_val * tau_t_i
                + 0.5 * diff_term
            )

            # directional derivative (ξ·∇)σ ∈ R^{2×2}:
            # Dsigma has shape (2,2,2) with last axis the direction component
            dir_sigma = jnp.einsum("ijk,k->ij", Dsigma, xi_i)  # (2,2)

            # r2 ∈ R^{2×2}
            r2 = (Jxi_i @ sigma_mat) - dir_sigma - tau_i * sigma_t - 0.5 * sigma_mat * tau_t_i

            if use_abs:
                total = total + jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))
            else:
                total = total + jnp.sum(r1 * r1) + jnp.sum(r2 * r2)

        return total

    _point_residual_jit = jax.jit(_point_residual)

    def _loss_impl(params_gen, txy_batch: jnp.ndarray):
        """
        txy_batch: (B,3) with columns [t, x, y].
        """
        if txy_batch.ndim != 2 or txy_batch.shape[1] != 3:
            raise ValueError(f"S6(2D) expects txy_batch shape (B,3)=[t,x,y]; got {txy_batch.shape}")
        per_point = jax.vmap(lambda z: _point_residual_jit(params_gen, z))(txy_batch)
        loss = jnp.mean(per_point)
        return loss, {"per_point": per_point}

    return jax.jit(_loss_impl)


## Loss 7 - SDE flow

In [ ]:
# ============================ S7 — Prolonged pushforward residual (2D SDE, tau/xi only) ============================
import jax
import jax.numpy as jnp

def make_s7_pushforward_coeff_loss_sde_only_2d(
    *,
    mu_fn,
    sig_fn,
    eps: float = 1e-2,
    num_steps: int = 1,
    sigma_floor: float = 1e-8,
    dt_neg_penalty: float = 100.0,  # kept for API compatibility; NOT USED
    activation: str = "tanh",
    normalize_tx=None,  # optional preprocessing: (t,x,y)->(t,x,y)
    jit: bool = True,
    # safety / clipping
    tau_clip: float = 5.0,
    xi_clip: float = 5.0,
    t_clip_lo: float = -1e6,
    t_clip_hi: float =  1e6,
    x_clip_abs: float = 50.0,
    y_clip_abs: float = 50.0,
):
    """
    S7 (trajectory-level) validity for 2D SDE symmetries (tau, xi only), via "after-flow" residual.

    This version REMOVES any penalty that discourages time inversion / reordering.
    Time reversal is allowed: no monotonicity constraint is imposed on t_push along trajectories.

    Requires a callable:
      eval_generators_tau_xi_jit(params_gen, t, x, y, activation=..., normalize_tx=...)
    returning:
      tau_all: (m,B)
      xi_all : (m,B,2)
    """

    if "eval_generators_tau_xi_jit" not in globals() or (not callable(globals()["eval_generators_tau_xi_jit"])):
        raise NameError(
            "S7(2D) requires a callable eval_generators_tau_xi_jit(params_gen, t, x, y, ...)"
        )
    eval_gen_tau_xi = globals()["eval_generators_tau_xi_jit"]

    eps = jnp.asarray(eps, dtype=jnp.float64)
    num_steps = int(num_steps)

    # finite-difference steps
    fd_t = jnp.asarray(1e-3, dtype=jnp.float64)
    fd_x = jnp.asarray(1e-3, dtype=jnp.float64)
    fd_y = jnp.asarray(1e-3, dtype=jnp.float64)

    def _to_sigma_mat(sig_raw):
        """
        Accept σ as:
          - (...,2)   => diagonal entries -> (...,2,2)
          - (...,2,2) => already matrix
        """
        if sig_raw.ndim >= 1 and sig_raw.shape[-1] == 2 and (sig_raw.ndim == 1 or sig_raw.shape[-2] != 2):
            return jnp.diag(sig_raw) if sig_raw.ndim == 1 else jax.vmap(jnp.diag)(sig_raw)
        if sig_raw.shape[-2:] == (2, 2):
            return sig_raw
        raise ValueError(f"sig_fn must return shape (...,2) or (...,2,2); got {sig_raw.shape}")

    def _rhs_diag_with_derivs(params_gen, t_stack, x_stack, y_stack):
        """
        Diagonal evaluations for each generator i at its own points + FD derivatives.

        Returns:
          tau    : (m,B)
          xi     : (m,B,2)
          tau_t  : (m,B)
          xi_t   : (m,B,2)
          xi_x   : (m,B,2)
          xi_y   : (m,B,2)
          xi_xx  : (m,B,2)
          xi_xy  : (m,B,2)
          xi_yy  : (m,B,2)
        """
        m, B = t_stack.shape
        t_flat = t_stack.reshape(-1)
        x_flat = x_stack.reshape(-1)
        y_flat = y_stack.reshape(-1)

        def diag_from_flat(t_in, x_in, y_in):
            tau_all, xi_all = eval_gen_tau_xi(
                params_gen, t_in, x_in, y_in, activation=activation, normalize_tx=normalize_tx
            )  # tau: (m,mB), xi: (m,mB,2)

            tau_blk = tau_all.reshape(m, m, B)
            xi_blk  = xi_all.reshape(m, m, B, 2)

            idx = jnp.arange(m, dtype=jnp.int32)
            tau_d = tau_blk[idx, idx, :]
            xi_d  = xi_blk[idx, idx, :, :]

            tau_d = jnp.nan_to_num(tau_d, nan=0.0, posinf=0.0, neginf=0.0)
            xi_d  = jnp.nan_to_num(xi_d,  nan=0.0, posinf=0.0, neginf=0.0)


            # tau_d = tau_clip * jnp.tanh(tau_d / tau_clip)
            # xi_d  = xi_clip  * jnp.tanh(xi_d  / xi_clip)

            return tau_d, xi_d

        tau0, xi0 = diag_from_flat(t_flat, x_flat, y_flat)

        tau_p, xi_p = diag_from_flat(t_flat + fd_t, x_flat, y_flat)
        tau_m, xi_m = diag_from_flat(t_flat - fd_t, x_flat, y_flat)
        tau_t = (tau_p - tau_m) / (2.0 * fd_t)
        xi_t  = (xi_p  - xi_m)  / (2.0 * fd_t)

        _, xi_xp = diag_from_flat(t_flat, x_flat + fd_x, y_flat)
        _, xi_xm = diag_from_flat(t_flat, x_flat - fd_x, y_flat)
        xi_x  = (xi_xp - xi_xm) / (2.0 * fd_x)
        xi_xx = (xi_xp - 2.0 * xi0 + xi_xm) / (fd_x * fd_x)

        _, xi_yp = diag_from_flat(t_flat, x_flat, y_flat + fd_y)
        _, xi_ym = diag_from_flat(t_flat, x_flat, y_flat - fd_y)
        xi_y  = (xi_yp - xi_ym) / (2.0 * fd_y)
        xi_yy = (xi_yp - 2.0 * xi0 + xi_ym) / (fd_y * fd_y)

        _, xi_pp = diag_from_flat(t_flat, x_flat + fd_x, y_flat + fd_y)
        _, xi_pm = diag_from_flat(t_flat, x_flat + fd_x, y_flat - fd_y)
        _, xi_mp = diag_from_flat(t_flat, x_flat - fd_x, y_flat + fd_y)
        _, xi_mm = diag_from_flat(t_flat, x_flat - fd_x, y_flat - fd_y)
        xi_xy = (xi_pp - xi_pm - xi_mp + xi_mm) / (4.0 * fd_x * fd_y)

        tau_t = jnp.nan_to_num(tau_t, nan=0.0, posinf=0.0, neginf=0.0)
        xi_t  = jnp.nan_to_num(xi_t,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_x  = jnp.nan_to_num(xi_x,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_y  = jnp.nan_to_num(xi_y,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_xx = jnp.nan_to_num(xi_xx, nan=0.0, posinf=0.0, neginf=0.0)
        xi_xy = jnp.nan_to_num(xi_xy, nan=0.0, posinf=0.0, neginf=0.0)
        xi_yy = jnp.nan_to_num(xi_yy, nan=0.0, posinf=0.0, neginf=0.0)

        return tau0, xi0, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_xy, xi_yy

    def _flow_heun_allgens(params_gen, t0, x0, y0):
        """
        Push point cloud under each generator, AND integrate prolonged (mu,sigma) along epsilon.

        Returns:
          t_push: (m,B)
          x_push: (m,B)
          y_push: (m,B)
          mu_pred: (m,B,2)
          sg_pred: (m,B,2,2)
        """
        tau0, _ = eval_gen_tau_xi(params_gen, t0, x0, y0, activation=activation, normalize_tx=normalize_tx)
        m = tau0.shape[0]
        B = t0.shape[0]

        tS = jnp.broadcast_to(t0[None, :], (m, B))
        xS = jnp.broadcast_to(x0[None, :], (m, B))
        yS = jnp.broadcast_to(y0[None, :], (m, B))

        t0c = jnp.clip(jnp.nan_to_num(t0, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        x0c = jnp.clip(jnp.nan_to_num(x0, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)
        y0c = jnp.clip(jnp.nan_to_num(y0, nan=0.0, posinf=0.0, neginf=0.0), -y_clip_abs, y_clip_abs)

        mu0 = jnp.nan_to_num(mu_fn(t0c, x0c, y0c), nan=0.0, posinf=0.0, neginf=0.0)
        sg0_raw = jnp.nan_to_num(sig_fn(t0c, x0c, y0c), nan=0.0, posinf=0.0, neginf=0.0)
        sg0 = _to_sigma_mat(sg0_raw)

        if sg0.ndim == 2:
            sg0 = jnp.broadcast_to(sg0[None, :, :], (B, 2, 2))
        sg0 = jnp.maximum(jnp.abs(sg0), sigma_floor)

        muS = jnp.broadcast_to(mu0[None, :, :], (m, B, 2))
        sgS = jnp.broadcast_to(sg0[None, :, :, :], (m, B, 2, 2))

        def pos_floor_mat(M):
            return sigma_floor + jax.nn.softplus(M - sigma_floor)

        def one_step(_, state):
            tS, xS, yS, muS, sgS = state

            tau, xi, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_xy, xi_yy = _rhs_diag_with_derivs(params_gen, tS, xS, yS)

            Jxi = jnp.stack([xi_x, xi_y], axis=-1)  # (m,B,2,2)
            a = jnp.einsum("...ij,...kj->...ik", sgS, sgS)

            a11 = a[..., 0, 0]
            a12 = a[..., 0, 1]
            a22 = a[..., 1, 1]
            diff_term = a11[..., None] * xi_xx + (2.0 * a12[..., None]) * xi_xy + a22[..., None] * xi_yy

            k1_t = tau
            k1_x = xi[..., 0]
            k1_y = xi[..., 1]

            k1_mu = xi_t + jnp.einsum("...ij,...j->...i", Jxi, muS) + 0.5 * diff_term - muS * tau_t[..., None]
            k1_sg = jnp.einsum("...ij,...jk->...ik", Jxi, sgS) - 0.5 * sgS * tau_t[..., None, None]

            t_pred  = tS  + eps * k1_t
            x_pred  = xS  + eps * k1_x
            y_pred  = yS  + eps * k1_y
            mu_pred = muS + eps * k1_mu
            sg_pred = pos_floor_mat(sgS + eps * k1_sg)

            tau2, xi2, tau_t2, xi_t2, xi_x2, xi_y2, xi_xx2, xi_xy2, xi_yy2 = _rhs_diag_with_derivs(params_gen, t_pred, x_pred, y_pred)

            Jxi2 = jnp.stack([xi_x2, xi_y2], axis=-1)
            a2 = jnp.einsum("...ij,...kj->...ik", sg_pred, sg_pred)

            a11b = a2[..., 0, 0]
            a12b = a2[..., 0, 1]
            a22b = a2[..., 1, 1]
            diff_term2 = a11b[..., None] * xi_xx2 + (2.0 * a12b[..., None]) * xi_xy2 + a22b[..., None] * xi_yy2

            k2_t = tau2
            k2_x = xi2[..., 0]
            k2_y = xi2[..., 1]
            k2_mu = xi_t2 + jnp.einsum("...ij,...j->...i", Jxi2, mu_pred) + 0.5 * diff_term2 - mu_pred * tau_t2[..., None]
            k2_sg = jnp.einsum("...ij,...jk->...ik", Jxi2, sg_pred) - 0.5 * sg_pred * tau_t2[..., None, None]

            t_new  = tS  + 0.5 * eps * (k1_t  + k2_t)
            x_new  = xS  + 0.5 * eps * (k1_x  + k2_x)
            y_new  = yS  + 0.5 * eps * (k1_y  + k2_y)
            mu_new = muS + 0.5 * eps * (k1_mu + k2_mu)
            sg_new = pos_floor_mat(sgS + 0.5 * eps * (k1_sg + k2_sg))

            return (t_new, x_new, y_new, mu_new, sg_new)

        tS, xS, yS, muS, sgS = jax.lax.fori_loop(0, num_steps, one_step, (tS, xS, yS, muS, sgS))
        return tS, xS, yS, muS, sgS

    def _loss_impl(params_gen, t_grid, xy_paths):
        t_grid   = jnp.asarray(t_grid, dtype=jnp.float64)
        xy_paths = jnp.asarray(xy_paths, dtype=jnp.float64)

        if xy_paths.ndim != 3 or xy_paths.shape[-1] != 2:
            raise ValueError(f"S7(2D) expects xy_paths shape (n_traj, N+1, 2); got {xy_paths.shape}")

        n_traj, Np1, _ = xy_paths.shape
        t_mat = jnp.broadcast_to(t_grid[None, :], (n_traj, Np1))

        # LEFT endpoints only
        t_left = t_mat[:, :-1].reshape(-1)
        x_left = xy_paths[:, :-1, 0].reshape(-1)
        y_left = xy_paths[:, :-1, 1].reshape(-1)
        B_left = t_left.shape[0]

        # prolong + push (left endpoints)
        t_push_L, x_push_L, y_push_L, mu_pred, sg_pred = _flow_heun_allgens(params_gen, t_left, x_left, y_left)

        # evaluate coefficients at pushed points
        tpc = jnp.clip(jnp.nan_to_num(t_push_L, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        xpc = jnp.clip(jnp.nan_to_num(x_push_L, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)
        ypc = jnp.clip(jnp.nan_to_num(y_push_L, nan=0.0, posinf=0.0, neginf=0.0), -y_clip_abs, y_clip_abs)

        m = t_push_L.shape[0]
        t_flat = tpc.reshape(-1)
        x_flat = xpc.reshape(-1)
        y_flat = ypc.reshape(-1)

        mu_eval = jnp.nan_to_num(mu_fn(t_flat, x_flat, y_flat), nan=0.0, posinf=0.0, neginf=0.0)
        sg_eval_raw = jnp.nan_to_num(sig_fn(t_flat, x_flat, y_flat), nan=0.0, posinf=0.0, neginf=0.0)
        sg_eval = _to_sigma_mat(sg_eval_raw)

        mu_eval = mu_eval.reshape(m, B_left, 2)
        if sg_eval.ndim == 2:
            sg_eval = jnp.broadcast_to(sg_eval[None, None, :, :], (m, B_left, 2, 2))
        elif sg_eval.ndim == 3:
            sg_eval = sg_eval.reshape(m, B_left, 2, 2)
        else:
            sg_eval = sg_eval.reshape(m, B_left, 2, 2)

        sg_eval = jnp.maximum(jnp.abs(sg_eval), sigma_floor)

        # residuals
        mu_mse = jnp.mean(jnp.sum((mu_pred - mu_eval) ** 2, axis=-1), axis=1)               # (m,)
        sg_mse = jnp.mean(jnp.sum((sg_pred - sg_eval) ** 2, axis=(-1, -2)), axis=1)        # (m,)

        # REMOVED: any penalty discouraging time inversion / reordering
        dt_neg_mean = jnp.zeros((m,), dtype=jnp.float64)

        per_gen_loss = mu_mse + sg_mse
        loss = jnp.mean(per_gen_loss)

        aux = {
            "per_gen_loss": per_gen_loss,
            "per_gen_stats": jnp.stack([mu_mse, sg_mse, dt_neg_mean], axis=1),  # (m,3) kept for compatibility
            "eps": eps,
            "num_steps": jnp.asarray(num_steps, dtype=jnp.int32),
        }
        return loss, aux

    return jax.jit(_loss_impl) if jit else _loss_impl


# Master Loss - SDE Symmetry

In [ ]:
# ============================ Master loss 1 (2D ONLY): weighted sum of L1–L7 ============================
# 2D pipeline ONLY: generator acts on z=(t,x,y) with
#   X_i = tau_i(t) ∂_t + xi_i(t,x,y)·∇_{x,y}
#
# Assumes in scope:
#   - params_sde, in_norm_sde, f_sigma_hat, cfg
#   - gen_cfg, params_gen
#   - t_norm_gen, tx_norm_gen
#   - tau_forward, xi_forward
#   - TX_gen (B,3) cloud for training batches (columns [t,x,y])
#   - (S7 data): either (t_surr, XY_surr) or (t, XY)  with XY shape (n_traj, N+1, 2)
#
# Defines/exports:
#   - LossWeights dataclass + loss_cfg
#   - mu_fn_2d, sig_mat_2d
#   - 2D losses s1..s7
#   - master_loss(params_gen, tx_batch, key=None) and master_loss_jit

from dataclasses import dataclass
import jax
import jax.numpy as jnp

# ----------------------- Loss weights / hyperparameters ----------------------

@dataclass
class LossWeights:
    # Algebraic structure terms
    w_s1_closure: float = 1.0   # L1: closure + constancy
    w_s2_jacobi:  float = 0.1   # L2: Jacobi identity
    w_s3_skew:    float = 0.1   # L3: skew-symmetry
    w_s4_bilin:   float = 0.1   # L4: bilinearity

    # Column independence
    w_s5_indep:   float = 0.1   # L5: functional independence

    # SDE determining equations
    w_s6_det:     float = 1.0   # L6: Ito determining equations (2D)

    # Finite-ε flow-validity
    w_s7_push:    float = 0.1   # L7: pushforward on (μ, B) (2D)

    # Generator weight decay
    weight_decay: float = 1e-6

    # S5 options
    s5_mode: str = "sigma"      # "sigma" or "corr_l2"
    s5_tau:  float = 0.8

    # S7 options
    s7_eps:   float = 1e-2
    s7_steps: int   = 1

loss_cfg = LossWeights()

# ----------------------- Basic checks (2D-only) ------------------------------

GEN_IN_DIM = int(jnp.asarray(tx_norm_gen.mean).shape[0])
assert GEN_IN_DIM == 3, f"2D-only master loss expects tx_norm_gen dim 3 (t,x,y). Got {GEN_IN_DIM}."

# ----------------------- Helpers: pytree L2 -----------------------

def l2_tree(params):
    return sum(jnp.sum(jnp.square(p)) for p in jax.tree_util.tree_leaves(params))

# ----------------------- 2D SDE surrogate wrappers: μ(t,x,y), B(t,x,y) --------

MU_CLIP  = 50.0
SIG_CLIP = 50.0

def _sde_forward_2d(t, x, y):
    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)
    y_arr = jnp.asarray(y, dtype=jnp.float64)
    t_b, x_b, y_b = jnp.broadcast_arrays(t_arr, x_arr, y_arr)
    txy = jnp.stack([t_b.ravel(), x_b.ravel(), y_b.ravel()], axis=-1)  # (B,3)

    txy_norm = in_norm_sde(txy)
    f_hat, s_hat = f_sigma_hat(
        params_sde,
        txy_norm,
        activation="tanh",
        sigma_min=cfg.sigma_min,
        learn_sigma_x=getattr(cfg, "learn_sigma_x", True),
    )
    f_hat = f_hat.reshape(t_b.shape + (2,))
    s_hat = s_hat.reshape(t_b.shape + (2,))
    return f_hat, s_hat

def mu_fn_2d(t, x, y):
    f_hat, _ = _sde_forward_2d(t, x, y)
    f_hat = jnp.nan_to_num(f_hat, nan=0.0, posinf=0.0, neginf=0.0)
    return jnp.clip(f_hat, -MU_CLIP, MU_CLIP)  # (...,2)

def sig_diag_2d(t, x, y):
    _, s_hat = _sde_forward_2d(t, x, y)
    s_hat = jnp.nan_to_num(s_hat, nan=0.0, posinf=0.0, neginf=0.0)
    return jnp.clip(s_hat, -SIG_CLIP, SIG_CLIP)  # (...,2)

def sig_mat_2d(t, x, y):
    s = sig_diag_2d(t, x, y)  # (...,2)
    z = jnp.zeros_like(s[..., 0])
    row0 = jnp.stack([s[..., 0], z], axis=-1)
    row1 = jnp.stack([z, s[..., 1]], axis=-1)
    return jnp.stack([row0, row1], axis=-2)  # (...,2,2)

# ----------------------- 2D generator evaluation helpers ---------------------

def _Xi_2d(params_tau_i, params_xi_i, z):
    # z=(t,x,y) -> (tau, xi_x, xi_y)
    t, x, y = z[0], z[1], z[2]

    t_arr = jnp.asarray([[t]], dtype=jnp.float64)
    t_norm = t_norm_gen(t_arr)
    tau = tau_forward(params_tau_i, t_norm, activation=gen_cfg.activation)[0, 0]

    txy_arr = jnp.asarray([[t, x, y]], dtype=jnp.float64)
    txy_norm = tx_norm_gen(txy_arr)
    xi = xi_forward(params_xi_i, txy_norm, activation=gen_cfg.activation)[0]  # (2,)

    xi = jnp.asarray(xi, dtype=jnp.float64)
    return jnp.array([tau, xi[0], xi[1]], dtype=jnp.float64)

def _ordered_pairs(m: int):
    idx = jnp.arange(m, dtype=jnp.int32)
    ii = jnp.repeat(idx, repeats=m-1)
    base = jnp.arange(m - 1, dtype=jnp.int32)
    i_col = idx[:, None]
    jj_mat = base + (base >= i_col).astype(jnp.int32)
    jj = jj_mat.reshape(-1)
    return ii, jj  # (K,), (K,)

# ----------------------- L1 (2D): closure + constancy ------------------------

def make_s1_lie_loss_2d(n_generators: int, rcond: float = 1e-2):
    ii, jj = _ordered_pairs(n_generators)
    reg = (jnp.asarray(rcond, dtype=jnp.float64) ** 2)

    def _at_point(params_gen_local, z):
        # F (m,3), J (m,3,3)
        F_list, J_list = [], []
        for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
            Xi = lambda zz: _Xi_2d(p_tau, p_xi, zz)
            Fi = Xi(z)
            Ji = jax.jacobian(Xi)(z)
            F_list.append(Fi)
            J_list.append(Ji)
        F = jnp.stack(F_list, axis=0)  # (m,3)
        J = jnp.stack(J_list, axis=0)  # (m,3,3)

        # brackets B (3,K)
        def br(p, q):
            return (J[q] @ F[p]) - (J[p] @ F[q])  # (3,)
        B = jax.vmap(br)(ii, jj).T  # (3,K)

        # project onto span(V) where V = F^T (3,m)
        V = F.T  # (3,m)
        G = (V.T @ V) + reg * jnp.eye(n_generators, dtype=jnp.float64)  # (m,m)
        RHS = V.T @ B                                                   # (m,K)
        C = jnp.linalg.solve(G, RHS)                                    # (m,K)
        P = V @ C                                                       # (3,K)
        E = B - P                                                       # (3,K)

        err = jnp.sum(E * E)
        return err, C

    def _loss_impl(params_gen_local, batch):
        errs, Cs = jax.vmap(lambda z: _at_point(params_gen_local, z))(batch)
        error_sum = jnp.sum(errs)
        C_var = jnp.var(Cs, axis=0)
        var_sum = jnp.sum(C_var)
        total = error_sum + var_sum
        aux = {"error_sum": error_sum, "var_sum": var_sum}
        return total, aux

    return jax.jit(_loss_impl)

# ----------------------- L3 (2D): skew-symmetry ------------------------------

def make_s3_skewsym_loss_2d(n_generators: int):
    pairs = [(i, j) for i in range(n_generators) for j in range(i + 1, n_generators)]
    if not pairs:
        def _zero(params_gen_local, batch):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((batch.shape[0],), dtype=jnp.float64), "num_pairs": 0}
        return jax.jit(_zero)

    pi = jnp.array([p[0] for p in pairs], dtype=jnp.int32)
    pj = jnp.array([p[1] for p in pairs], dtype=jnp.int32)

    def _point(params_gen_local, z):
        F_list, J_list = [], []
        for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
            Xi = lambda zz: _Xi_2d(p_tau, p_xi, zz)
            F_list.append(Xi(z))
            J_list.append(jax.jacobian(Xi)(z))
        F = jnp.stack(F_list, axis=0)  # (m,3)
        J = jnp.stack(J_list, axis=0)  # (m,3,3)

        def br(p, q):
            return (J[q] @ F[p]) - (J[p] @ F[q])

        def one(i, j):
            r = br(i, j) + br(j, i)
            return jnp.sum(jnp.abs(r))

        return jnp.sum(jax.vmap(one)(pi, pj))

    _pl = jax.jit(_point)

    def _loss_impl(params_gen_local, batch):
        per_point = jax.vmap(lambda z: _pl(params_gen_local, z))(batch)
        total = jnp.sum(per_point)

        aux = {"per_point": per_point, "num_pairs": int(pi.shape[0])}

        total = jnp.sum(total)

        return total, aux

    return jax.jit(_loss_impl)

# ----------------------- L4 (2D): bilinearity --------------------------------

def make_s4_bilinearity_loss_2d(n_generators: int, num_cc: int = 4, cc_list=None, normalize: bool = True):
    triples = [(i, j, k) for i in range(n_generators) for j in range(i + 1, n_generators) for k in range(j + 1, n_generators)]
    if not triples:
        def _zero(params_gen_local, batch, key=None):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((batch.shape[0],), dtype=jnp.float64)}
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    perms6 = jnp.array([[0,1,2],[0,2,1],[1,0,2],[1,2,0],[2,0,1],[2,1,0]], dtype=jnp.int32)

    def _point(params_gen_local, z, cc):
        F_list, J_list = [], []
        for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
            Xi = lambda zz: _Xi_2d(p_tau, p_xi, zz)
            F_list.append(Xi(z))
            J_list.append(jax.jacobian(Xi)(z))
        F = jnp.stack(F_list, axis=0)  # (m,3)
        J = jnp.stack(J_list, axis=0)  # (m,3,3)

        def br(p, q):
            return (J[q] @ F[p]) - (J[p] @ F[q])  # (3,)

        def triple_terms(i, j, k):
            inds = jnp.array([i, j, k], dtype=jnp.int32)

            def one_perm(p):
                u, v, w = inds[p[0]], inds[p[1]], inds[p[2]]
                fu, fv, fw = F[u], F[v], F[w]
                Ju, Jv, Jw = J[u], J[v], J[w]

                def one_cc(cpair):
                    c, cp = cpair[0], cpair[1]

                    f_uv = c * fu + cp * fv
                    J_uv = c * Ju + cp * Jv

                    f_vw = c * fv + cp * fw
                    J_vw = c * Jv + cp * Jw

                    term1 = (Jw @ f_uv) - (J_uv @ fw)
                    rhs1  = c * br(u, w) + cp * br(v, w)
                    r1 = term1 - rhs1

                    term2 = (J_vw @ fu) - (Ju @ f_vw)
                    rhs2  = c * br(u, v) + cp * br(u, w)
                    r2 = term2 - rhs2

                    if normalize:
                        denom = jnp.abs(c) + jnp.abs(cp) + 1e-12
                        r1 = r1 / denom
                        r2 = r2 / denom

                    return jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))

                vals_cc = jax.vmap(one_cc)(cc)
                return jnp.mean(vals_cc)

            vals = jax.vmap(one_perm)(perms6)
            return jnp.sum(vals)

        per_tr = jax.vmap(triple_terms)(tri_i, tri_j, tri_k)
        return jnp.sum(per_tr)

    _pl = jax.jit(_point)

    def _loss_impl(params_gen_local, batch, key=None):
        if cc_list is not None:
            cc = jnp.asarray(cc_list, dtype=jnp.float64)
        else:
            key = jax.random.PRNGKey(0) if key is None else key
            cc = jax.random.uniform(key, (num_cc, 2), minval=-1.0, maxval=1.0, dtype=jnp.float64)

        per_point = jax.vmap(lambda z: _pl(params_gen_local, z, cc))(batch)
        total = jnp.sum(per_point)
        aux = {"per_point": per_point, "num_triples": int(tri_i.shape[0]), "num_cc": int(cc.shape[0])}
        return total, aux

    return jax.jit(_loss_impl)

# ----------------------- L2 (2D): Jacobi identity (nested brackets via autograd) ----
# FIXED: avoids Python list indexing with traced indices via lax.switch.

def make_s2_jacobi_loss_2d(n_generators: int):
    triples = [(i, j, k) for i in range(n_generators) for j in range(i + 1, n_generators) for k in range(j + 1, n_generators)]
    if not triples:
        def _zero(params_gen_local, batch):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((batch.shape[0],), dtype=jnp.float64), "num_triples": 0}
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    def _tree_take(seq, idx):
        """
        JIT-safe selection from a Python list/tuple of pytrees using a traced integer idx.
        """
        idx = jnp.asarray(idx, dtype=jnp.int32)
        branches = [lambda _, s=s: s for s in seq]   # <-- IMPORTANT: bind s
        return jax.lax.switch(idx, branches, operand=None)


    def _Xi_fun(params_tau_i, params_xi_i):
        return lambda zz: _Xi_2d(params_tau_i, params_xi_i, zz)

    def _bracket_fun(params_gen_local, p, q):
        p_tau = _tree_take(params_gen_local["tau"], p)
        p_xi  = _tree_take(params_gen_local["xi"],  p)
        q_tau = _tree_take(params_gen_local["tau"], q)
        q_xi  = _tree_take(params_gen_local["xi"],  q)

        Xp = _Xi_fun(p_tau, p_xi)
        Xq = _Xi_fun(q_tau, q_xi)

        def B(zz):
            Jp = jax.jacobian(Xp)(zz)
            Jq = jax.jacobian(Xq)(zz)
            fp = Xp(zz)
            fq = Xq(zz)

            return (Jq @ fp) - (Jp @ fq)
        return B

    def _double_bracket(params_gen_local, a, b, c, z):
        # [[Xa,Xb],Xc](z)
        Bab = _bracket_fun(params_gen_local, a, b)

        c_tau = _tree_take(params_gen_local["tau"], c)
        c_xi  = _tree_take(params_gen_local["xi"],  c)
        Xc = _Xi_fun(c_tau, c_xi)

        Jc  = jax.jacobian(Xc)(z)
        Jab = jax.jacobian(Bab)(z)
        fab = Bab(z)
        fc  = Xc(z)
        return (Jc @ fab) - (Jab @ fc)

    def _jacobi(params_gen_local, i, j, k, z):
        r = _double_bracket(params_gen_local, i, j, k, z) \
          + _double_bracket(params_gen_local, j, k, i, z) \
          + _double_bracket(params_gen_local, k, i, j, z)
        return jnp.sum(jnp.abs(r))

    def _point(params_gen_local, z):
        vals = jax.vmap(lambda a, b, c: _jacobi(params_gen_local, a, b, c, z))(tri_i, tri_j, tri_k)
        return jnp.sum(vals)

    _pl = jax.jit(_point)

    def _loss_impl(params_gen_local, batch):
        per_point = jax.vmap(lambda z: _pl(params_gen_local, z))(batch)
        total = jnp.sum(per_point)

        aux = {"loss": total, "per_point": per_point, "num_triples": int(tri_i.shape[0])}

        return total, aux


    return jax.jit(_loss_impl)

# ----------------------- L5 (2D): column independence ------------------------

def make_s5_column_independence_loss_2d(n_generators: int, *, mode: str = "sigma", tau: float = 0.0, eps: float = 1e-12):
    mode = "sigma" if mode == "sigma" else "corr_l2"
    mode_code = 0 if mode == "sigma" else 1

    def _eval_F_batch(params_gen_local, batch):
        def one(z):
            F_list = []
            for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
                F_list.append(_Xi_2d(p_tau, p_xi, z))
            return jnp.stack(F_list, axis=0)  # (m,3)
        return jax.vmap(one)(batch)  # (B,m,3)

    def _loss_impl(params_gen_local, batch):
        Fm3 = _eval_F_batch(params_gen_local, batch)                 # (B,m,3)
        A = jnp.transpose(Fm3, (0, 2, 1)).reshape(-1, n_generators)  # (3B,m)

        col_norms = jnp.linalg.norm(A, axis=0) + eps
        Ahat = A / col_norms
        G = Ahat.T @ Ahat

        if mode_code == 0:
            lam = jnp.linalg.eigvalsh(G)
            lam_min = jnp.clip(jnp.min(lam), 0.0, None)
            sigma_min = jnp.sqrt(lam_min)
            loss = jnp.maximum(0.0, jnp.asarray(tau, dtype=G.dtype) - sigma_min)
            aux = {"sigma_min": sigma_min}
        else:
            I = jnp.eye(G.shape[0], dtype=G.dtype)
            off = G - I
            off = off - jnp.diag(jnp.diag(off))
            loss = jnp.sum(off * off)
            aux = {"gram_diag_mean": jnp.mean(jnp.diag(G))}
        return loss, aux

    return jax.jit(_loss_impl)

# ----------------------- L6 (2D): Ito determining equations ------------------

def make_s6_commutator_loss_ito_2d(*, mu_fn, sig_mat_fn, use_abs: bool = False):
    def _tau(params_tau_i, t_scalar):
        t_arr = jnp.asarray([[t_scalar]], dtype=jnp.float64)
        t_norm = t_norm_gen(t_arr)
        return tau_forward(params_tau_i, t_norm, activation=gen_cfg.activation)[0, 0]

    def _xi(params_xi_i, t_scalar, x_scalar, y_scalar):
        txy_arr = jnp.asarray([[t_scalar, x_scalar, y_scalar]], dtype=jnp.float64)
        txy_norm = tx_norm_gen(txy_arr)
        v = xi_forward(params_xi_i, txy_norm, activation=gen_cfg.activation)[0]  # (2,)
        return jnp.asarray(v, dtype=jnp.float64)

    def _point(params_gen_local, z):
        t, x, y = z[0], z[1], z[2]

        mu = mu_fn(t, x, y)            # (2,)
        B  = sig_mat_fn(t, x, y)       # (2,2)
        Q  = B @ B.T                   # (2,2)

        mu_t = jax.jacobian(lambda tt: mu_fn(tt, x, y))(t)  # (2,)
        mu_xy = jax.jacobian(lambda zz: mu_fn(t, zz[0], zz[1]))(jnp.array([x, y], dtype=jnp.float64))  # (2,2)

        B_t = jax.jacobian(lambda tt: sig_mat_fn(tt, x, y))(t)  # (2,2)
        B_xy = jax.jacobian(lambda zz: sig_mat_fn(t, zz[0], zz[1]))(jnp.array([x, y], dtype=jnp.float64))  # (2,2,2)

        total = jnp.asarray(0.0, dtype=jnp.float64)

        for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
            tau0 = _tau(p_tau, t)
            tau_t = jax.grad(lambda tt: _tau(p_tau, tt))(t)

            xi0 = _xi(p_xi, t, x, y)                                  # (2,)
            xi_t = jax.jacobian(lambda tt: _xi(p_xi, tt, x, y))(t)     # (2,)

            def xi_xy(zz):
                return _xi(p_xi, t, zz[0], zz[1])  # (2,)

            Jxi = jax.jacobian(xi_xy)(jnp.array([x, y], dtype=jnp.float64))  # (2,2)
            Hxi = jax.jacobian(lambda zz: jax.jacobian(xi_xy)(zz))(jnp.array([x, y], dtype=jnp.float64))  # (2,2,2)

            mu_grad_xi = Jxi @ mu       # (2,)
            grad_mu_xi = mu_xy @ xi0    # (2,)
            quad = 0.5 * jnp.einsum("ab,iab->i", Q, Hxi)  # (2,)

            r1 = xi_t + mu_grad_xi - grad_mu_xi - tau0 * mu_t - tau_t * mu + quad  # (2,)

            term1 = Jxi @ B
            dirB = (B_xy[..., 0] * xi0[0]) + (B_xy[..., 1] * xi0[1])  # (2,2)
            r2 = term1 - dirB - tau0 * B_t - 0.5 * tau_t * B          # (2,2)

            if use_abs:
                total = total + jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))
            else:
                total = total + jnp.sum(r1 * r1) + jnp.sum(r2 * r2)

        return total

    _point_jit = jax.jit(_point)

    def _loss_impl(params_gen_local, batch):
        per_point = jax.vmap(lambda z: _point_jit(params_gen_local, z))(batch)
        loss = jnp.mean(per_point)
        aux = {"per_point": per_point}
        return loss, aux

    return jax.jit(_loss_impl)

# ----------------------- L7 (2D): pushforward on (mu, B) ---------------------

def make_s7_pushforward_coeff_loss_sde_2d(
    *,
    mu_fn,
    sig_mat_fn,
    eps: float = 1e-2,
    num_steps: int = 1,
    sigma_floor: float = 1e-8,
    dt_neg_penalty: float = 10.0,
    fd_t: float = 1e-3,
    fd_x: float = 1e-3,
    fd_y: float = 1e-3,
    tau_clip: float = 5.0,
    xi_clip: float = 5.0,
    t_clip_lo: float = -1e6,
    t_clip_hi: float =  1e6,
    x_clip_abs: float = 50.0,
    y_clip_abs: float = 50.0,
):
    eps = jnp.asarray(eps, dtype=jnp.float64)
    num_steps = int(num_steps)
    fd_t = jnp.asarray(fd_t, dtype=jnp.float64)
    fd_x = jnp.asarray(fd_x, dtype=jnp.float64)
    fd_y = jnp.asarray(fd_y, dtype=jnp.float64)

    def _eval_diag_tau_xi(params_gen_local, t_stack, x_stack, y_stack):
        m, B = t_stack.shape
        t_flat = t_stack.reshape(-1)
        x_flat = x_stack.reshape(-1)
        y_flat = y_stack.reshape(-1)

        def diag_from_flat(t_in, x_in, y_in):
            tau_rows = []
            xi_rows  = []
            for p_tau, p_xi in zip(params_gen_local["tau"], params_gen_local["xi"]):
                t_arr = jnp.asarray(t_in, dtype=jnp.float64).reshape(-1, 1)
                t_norm = t_norm_gen(t_arr)
                tau_all = tau_forward(p_tau, t_norm, activation=gen_cfg.activation).reshape(-1)  # (mB,)

                txy = jnp.stack([t_in, x_in, y_in], axis=1)  # (mB,3)
                txy_norm = tx_norm_gen(txy)
                xi_all = xi_forward(p_xi, txy_norm, activation=gen_cfg.activation)               # (mB,2)
                xi_all = jnp.asarray(xi_all, dtype=jnp.float64)

                tau_rows.append(tau_all)
                xi_rows.append(xi_all)

            tau_all = jnp.stack(tau_rows, axis=0)          # (m, mB)
            xi_all  = jnp.stack(xi_rows, axis=0)           # (m, mB, 2)

            tau_blk = tau_all.reshape(m, m, B)             # (m,m,B)
            xi_blk  = xi_all.reshape(m, m, B, 2)           # (m,m,B,2)

            idx = jnp.arange(m, dtype=jnp.int32)
            tau_diag = tau_blk[idx, idx, :]                # (m,B)
            xi_diag  = xi_blk[idx, idx, :, :]              # (m,B,2)

            tau_diag = jnp.nan_to_num(tau_diag, nan=0.0, posinf=0.0, neginf=0.0)
            xi_diag  = jnp.nan_to_num(xi_diag,  nan=0.0, posinf=0.0, neginf=0.0)

            tau_diag = tau_clip * jnp.tanh(tau_diag / tau_clip)
            xi_diag  = xi_clip  * jnp.tanh(xi_diag  / xi_clip)
            return tau_diag, xi_diag

        tau0, xi0 = diag_from_flat(t_flat, x_flat, y_flat)

        tau_p, xi_p = diag_from_flat(t_flat + fd_t, x_flat, y_flat)
        tau_m, xi_m = diag_from_flat(t_flat - fd_t, x_flat, y_flat)
        tau_t = (tau_p - tau_m) / (2.0 * fd_t)              # (m,B)
        xi_t  = (xi_p  - xi_m)  / (2.0 * fd_t)              # (m,B,2)

        _, xi_xp = diag_from_flat(t_flat, x_flat + fd_x, y_flat)
        _, xi_xm = diag_from_flat(t_flat, x_flat - fd_x, y_flat)
        xi_x = (xi_xp - xi_xm) / (2.0 * fd_x)               # (m,B,2)
        xi_xx = (xi_xp - 2.0 * xi0 + xi_xm) / (fd_x * fd_x) # (m,B,2)

        _, xi_yp = diag_from_flat(t_flat, x_flat, y_flat + fd_y)
        _, xi_ym = diag_from_flat(t_flat, x_flat, y_flat - fd_y)
        xi_y = (xi_yp - xi_ym) / (2.0 * fd_y)               # (m,B,2)
        xi_yy = (xi_yp - 2.0 * xi0 + xi_ym) / (fd_y * fd_y) # (m,B,2)

        tau_t = jnp.nan_to_num(tau_t, nan=0.0, posinf=0.0, neginf=0.0)
        xi_t  = jnp.nan_to_num(xi_t,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_x  = jnp.nan_to_num(xi_x,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_y  = jnp.nan_to_num(xi_y,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_xx = jnp.nan_to_num(xi_xx, nan=0.0, posinf=0.0, neginf=0.0)
        xi_yy = jnp.nan_to_num(xi_yy, nan=0.0, posinf=0.0, neginf=0.0)

        return tau0, xi0, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_yy

    def _flow_heun_allgens(params_gen_local, t0, x0, y0):
        t0 = jnp.asarray(t0, dtype=jnp.float64)
        x0 = jnp.asarray(x0, dtype=jnp.float64)
        y0 = jnp.asarray(y0, dtype=jnp.float64)

        m = int(gen_cfg.n_generators)
        Bn = int(t0.shape[0])

        tS = jnp.broadcast_to(t0[None, :], (m, Bn))
        xS = jnp.broadcast_to(x0[None, :], (m, Bn))
        yS = jnp.broadcast_to(y0[None, :], (m, Bn))

        t0c = jnp.clip(jnp.nan_to_num(t0, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        x0c = jnp.clip(jnp.nan_to_num(x0, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)
        y0c = jnp.clip(jnp.nan_to_num(y0, nan=0.0, posinf=0.0, neginf=0.0), -y_clip_abs, y_clip_abs)

        mu0 = jnp.nan_to_num(mu_fn(t0c, x0c, y0c), nan=0.0, posinf=0.0, neginf=0.0)  # (B,2)
        B0  = jnp.nan_to_num(sig_mat_fn(t0c, x0c, y0c), nan=0.0, posinf=0.0, neginf=0.0)  # (B,2,2)

        d0 = jnp.maximum(jnp.abs(B0[:, 0, 0]), sigma_floor)
        d1 = jnp.maximum(jnp.abs(B0[:, 1, 1]), sigma_floor)
        z = jnp.zeros((Bn,), dtype=jnp.float64)
        B0 = jnp.stack([jnp.stack([d0, z], axis=1),
                        jnp.stack([z,  d1], axis=1)], axis=1)  # (B,2,2)

        muS = jnp.broadcast_to(mu0[None, :, :], (m, Bn, 2))
        BS  = jnp.broadcast_to(B0[None, :, :, :], (m, Bn, 2, 2))

        def _pos_floor_diag(Bmat):
            d0 = sigma_floor + jax.nn.softplus(Bmat[..., 0, 0] - sigma_floor)
            d1 = sigma_floor + jax.nn.softplus(Bmat[..., 1, 1] - sigma_floor)
            z = jnp.zeros_like(d0)
            return jnp.stack([jnp.stack([d0, z], axis=-1),
                              jnp.stack([z,  d1], axis=-1)], axis=-2)

        def one_step(_, state):
            tS, xS, yS, muS, BS = state

            tau, xi, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_yy = _eval_diag_tau_xi(params_gen_local, tS, xS, yS)

            k1_t = tau
            k1_x = xi[..., 0]
            k1_y = xi[..., 1]

            J00 = xi_x[..., 0]; J01 = xi_y[..., 0]
            J10 = xi_x[..., 1]; J11 = xi_y[..., 1]
            Jxi = jnp.stack([jnp.stack([J00, J01], axis=-1),
                             jnp.stack([J10, J11], axis=-1)], axis=-2)  # (m,B,2,2)

            Q = BS @ jnp.swapaxes(BS, -1, -2)
            quad = 0.5 * (Q[..., 0, 0][..., None] * xi_xx + Q[..., 1, 1][..., None] * xi_yy)

            k1_mu = xi_t + jnp.einsum("...ij,...j->...i", Jxi, muS) + quad - (muS * tau_t[..., None])
            k1_B  = jnp.einsum("...ij,...jk->...ik", Jxi, BS) - 0.5 * tau_t[..., None, None] * BS

            tP  = tS  + eps * k1_t
            xP  = xS  + eps * k1_x
            yP  = yS  + eps * k1_y
            muP = muS + eps * k1_mu
            BP  = _pos_floor_diag(BS + eps * k1_B)

            tau2, xi2, tau_t2, xi_t2, xi_x2, xi_y2, xi_xx2, xi_yy2 = _eval_diag_tau_xi(params_gen_local, tP, xP, yP)

            k2_t = tau2
            k2_x = xi2[..., 0]
            k2_y = xi2[..., 1]

            J00 = xi_x2[..., 0]; J01 = xi_y2[..., 0]
            J10 = xi_x2[..., 1]; J11 = xi_y2[..., 1]
            Jxi2 = jnp.stack([jnp.stack([J00, J01], axis=-1),
                              jnp.stack([J10, J11], axis=-1)], axis=-2)

            Q2 = BP @ jnp.swapaxes(BP, -1, -2)
            quad2 = 0.5 * (Q2[..., 0, 0][..., None] * xi_xx2 + Q2[..., 1, 1][..., None] * xi_yy2)

            k2_mu = xi_t2 + jnp.einsum("...ij,...j->...i", Jxi2, muP) + quad2 - (muP * tau_t2[..., None])
            k2_B  = jnp.einsum("...ij,...jk->...ik", Jxi2, BP) - 0.5 * tau_t2[..., None, None] * BP

            tN  = tS  + 0.5 * eps * (k1_t  + k2_t)
            xN  = xS  + 0.5 * eps * (k1_x  + k2_x)
            yN  = yS  + 0.5 * eps * (k1_y  + k2_y)
            muN = muS + 0.5 * eps * (k1_mu + k2_mu)
            BN  = _pos_floor_diag(BS + 0.5 * eps * (k1_B + k2_B))

            return (tN, xN, yN, muN, BN)

        return jax.lax.fori_loop(0, int(num_steps), one_step, (tS, xS, yS, muS, BS))

    def _loss_impl(params_gen_local, t_grid, xy_paths):
        t_grid = jnp.asarray(t_grid, dtype=jnp.float64)
        xy_paths = jnp.asarray(xy_paths, dtype=jnp.float64)

        n_traj, Np1, two = xy_paths.shape
        assert two == 2, f"xy_paths last dim must be 2; got {two}"
        N = Np1 - 1

        t_mat = jnp.broadcast_to(t_grid[None, :], (n_traj, Np1))

        t_left = t_mat[:, :-1].reshape(-1)
        x_left = xy_paths[:, :-1, 0].reshape(-1)
        y_left = xy_paths[:, :-1, 1].reshape(-1)
        B_left = int(t_left.shape[0])

        tP, xP, yP, mu_pred, B_pred = _flow_heun_allgens(params_gen_local, t_left, x_left, y_left)

        tpc = jnp.clip(jnp.nan_to_num(tP, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        xpc = jnp.clip(jnp.nan_to_num(xP, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)
        ypc = jnp.clip(jnp.nan_to_num(yP, nan=0.0, posinf=0.0, neginf=0.0), -y_clip_abs, y_clip_abs)

        m = int(tP.shape[0])
        mu_eval = mu_fn(tpc.reshape(-1), xpc.reshape(-1), ypc.reshape(-1)).reshape(m, B_left, 2)
        B_eval  = sig_mat_fn(tpc.reshape(-1), xpc.reshape(-1), ypc.reshape(-1)).reshape(m, B_left, 2, 2)

        d0 = jnp.maximum(jnp.abs(B_eval[..., 0, 0]), sigma_floor)
        d1 = jnp.maximum(jnp.abs(B_eval[..., 1, 1]), sigma_floor)
        z = jnp.zeros_like(d0)
        B_eval = jnp.stack([jnp.stack([d0, z], axis=-1),
                            jnp.stack([z,  d1], axis=-1)], axis=-2)

        mu_mse = jnp.mean((mu_pred - mu_eval) ** 2, axis=(1, 2))
        B_mse  = jnp.mean((B_pred  - B_eval)  ** 2, axis=(1, 2, 3))

        t_full = t_mat.reshape(-1)
        x_full = xy_paths[..., 0].reshape(-1)
        y_full = xy_paths[..., 1].reshape(-1)
        tP_full, _, _, _, _ = _flow_heun_allgens(params_gen_local, t_full, x_full, y_full)

        tP_full = tP_full.reshape(m, n_traj, Np1)
        dtp = tP_full[:, :, 1:] - tP_full[:, :, :-1]
        dt_neg = jax.nn.softplus(-dtp)
        dt_neg_mean = jnp.mean(dt_neg, axis=(1, 2))

        per_gen_loss = mu_mse + B_mse + dt_neg_penalty * dt_neg_mean
        loss = jnp.mean(per_gen_loss)

        aux = {
            "per_gen_loss": per_gen_loss,
            "per_gen_stats": jnp.stack([mu_mse, B_mse, dt_neg_mean], axis=1),
            "eps": jnp.asarray(eps, dtype=jnp.float64),
            "num_steps": jnp.asarray(num_steps, dtype=jnp.int32),
        }
        return loss, aux

    return jax.jit(_loss_impl)

# ----------------------- Instantiate losses (2D-only) ------------------------

s1_lie_loss     = make_s1_lie_loss_2d(n_generators=gen_cfg.n_generators)
s2_jacobi_loss  = make_s2_jacobi_loss_2d(n_generators=gen_cfg.n_generators)
s3_skew_loss    = make_s3_skewsym_loss_2d(n_generators=gen_cfg.n_generators)
s4_bilin_loss   = make_s4_bilinearity_loss_2d(n_generators=gen_cfg.n_generators, num_cc=4, cc_list=None, normalize=True)
s5_indep_loss   = make_s5_column_independence_loss_2d(n_generators=gen_cfg.n_generators, mode=loss_cfg.s5_mode, tau=loss_cfg.s5_tau, eps=1e-12)
s6_det_loss     = make_s6_commutator_loss_ito_2d(mu_fn=mu_fn_2d, sig_mat_fn=sig_mat_2d, use_abs=False)

s7_push_loss_raw = make_s7_pushforward_coeff_loss_sde_2d(
    mu_fn=mu_fn_2d,
    sig_mat_fn=sig_mat_2d,
    eps=loss_cfg.s7_eps,
    num_steps=loss_cfg.s7_steps,
    sigma_floor=1e-6,
    dt_neg_penalty=10.0,
    tau_clip=5.0,
    xi_clip=5.0,
    x_clip_abs=25.0,
    y_clip_abs=25.0,
)

# ----------------------- S7 data (auto-define once, no extra cell) -----------

if ("t_s7" not in globals()) or ("xy_paths_s7" not in globals()):
    if ("t_surr" in globals()) and ("XY_surr" in globals()):
        t_src = globals()["t_surr"]
        XY_src = globals()["XY_surr"]
    else:
        t_src = globals().get("t", None)
        XY_src = globals().get("XY", None)
    assert t_src is not None and XY_src is not None, "Need (t_surr,XY_surr) or (t,XY) for 2D S7."

    n_traj_use = int(min(64, XY_src.shape[0]))
    Np1_use    = int(min(401, XY_src.shape[1]))
    t_s7 = jnp.asarray(t_src[:Np1_use], dtype=jnp.float64)
    xy_paths_s7 = jnp.asarray(XY_src[:n_traj_use, :Np1_use, :], dtype=jnp.float64)  # (n_traj,T,2)
    globals().update({"t_s7": t_s7, "xy_paths_s7": xy_paths_s7})

def s7_push_loss(params_gen_local, tx_batch, t_s7=None, xy_paths=None):
    t_grid = globals()["t_s7"] if t_s7 is None else t_s7
    paths  = globals()["xy_paths_s7"] if xy_paths is None else xy_paths
    return s7_push_loss_raw(params_gen_local, t_grid, paths)

# ----------------------- Master loss ----------------------------------------

def master_loss(params_gen, tx_batch: jnp.ndarray, key=None, t_s7=None, xy_paths=None):
    """
    2D-only master loss: tx_batch is (B,3) with columns [t,x,y].
    """
    tx_batch = jnp.asarray(tx_batch, dtype=jnp.float64)
    assert tx_batch.ndim == 2 and tx_batch.shape[1] == 3, f"Expected tx_batch (B,3); got {tx_batch.shape}"

    # L1..L7 (all called with correct signatures)
    L1, aux1 = s1_lie_loss(params_gen, tx_batch)
    L2, aux2 = s2_jacobi_loss(params_gen, tx_batch)
    L3, aux3 = s3_skew_loss(params_gen, tx_batch)
    L4, aux4 = s4_bilin_loss(params_gen, tx_batch, key)   # <- uses key for random cc
    L5, aux5 = s5_indep_loss(params_gen, tx_batch)
    L6, aux6 = s6_det_loss(params_gen, tx_batch)
    L7, aux7 = s7_push_loss(params_gen, tx_batch, t_s7=t_s7, xy_paths=xy_paths)

    wd = loss_cfg.weight_decay * l2_tree(params_gen)

    total = (
        loss_cfg.w_s1_closure * L1 +
        loss_cfg.w_s2_jacobi  * L2 +
        loss_cfg.w_s3_skew    * L3 +
        loss_cfg.w_s4_bilin   * L4 +
        loss_cfg.w_s5_indep   * L5 +
        loss_cfg.w_s6_det     * L6 +
        loss_cfg.w_s7_push    * L7 +
        wd
    )

    aux = {
        "L1": {"loss": L1, **aux1},
        "L2": {"loss": L2, **aux2},
        "L3": {"loss": L3, **aux3},
        "L4": {"loss": L4, **aux4},
        "L5": {"loss": L5, **aux5},
        "L6": {"loss": L6, **aux6},
        "L7": {"loss": L7, **aux7},
        "weight_decay": wd,
        "total": total,
    }
    return total, aux


master_loss_jit = jax.jit(master_loss)

globals().update({
    "loss_cfg": loss_cfg,
    "master_loss": master_loss,
    "master_loss_jit": master_loss_jit,
    "mu_fn_2d": mu_fn_2d,
    "sig_mat_2d": sig_mat_2d,
})


# Training

In [ ]:
# ============================ Generator training loop (Stage 2) — 2D ONLY ============================
# This cell ASSUMES the CURRENT pipeline is 2D SDE / 2D FP:
#   - TX_gen is (N,3) with columns [t, x, y]
#   - master_loss(params_gen, tx_batch, key) is the *2D* master (FP or SDE)
#   - params_gen is a pytree with ONLY float leaves (no int dtypes)
#
# In scope from earlier cells:
#   - gen_cfg, params_gen, TX_gen, key_main
#   - init_mlp_params
#   - optax
#   - master_loss (and master_loss_jit optional)

from dataclasses import dataclass
import numpy as np
import jax
import jax.numpy as jnp
import optax

# ---------- training config ----------
@dataclass
class GenTrainConfig2D:
    steps: int = 3000
    batch_size: int = 1024
    lr: float = 1e-4
    print_every: int = 10

gen_train_cfg = GenTrainConfig2D()

# ---------- enforce 2D dataset shape ----------
TX_gen_np = np.asarray(TX_gen, dtype=np.float64)
assert TX_gen_np.ndim == 2 and TX_gen_np.shape[1] == 3, (
    f"2D-only training expects TX_gen shape (N,3)=[t,x,y], got {TX_gen_np.shape}"
)

# Check the weights for L2 and L3 in loss_cfg
_cfg = globals().get("loss_cfg_fp_2d", globals().get("loss_cfg", None))
print(f"Jacobi weight (L2): {_cfg.w_s2_jacobi}")
print(f"Skew-symmetry weight (L3): {_cfg.w_s3_skew}")



mask = np.isfinite(TX_gen_np).all(axis=1)
TX_gen_np = TX_gen_np[mask]
N_tx = TX_gen_np.shape[0]
assert N_tx > 0, "All TX_gen points are non-finite after filtering."

# Inspect the input data TX_gen
print(f"TX_gen shape: {TX_gen.shape}")
print(f"First few samples of TX_gen: {TX_gen[:5]}")


# ---------- (optional) re-init if int leaves leaked ----------
def _has_integer_leaves(pytree):
    leaves = jax.tree_util.tree_leaves(pytree)
    return any(
        isinstance(a, jnp.ndarray) and jnp.issubdtype(a.dtype, jnp.integer)
        for a in leaves
    )

def _reinit_params_gen_2d(key, gen_cfg):
    """
    2D-only generator params initializer.
    Assumes 2D generator parameterization:
      - tau: scalar output (1)
      - xi : 2-vector output (2)  [xi_x, xi_y]
      - beta: scalar output (1)   (if FP/SDE setup uses it)
    NOTE: input to xi/beta nets remains (t,x) or (t,x,y).
          This training loop does NOT change that; it only re-inits with the SAME MLP helper.
          If xi net expects 3 inputs, change sizes[0] below to 3 accordingly.
    """
    assert "init_mlp_params" in globals(), "Missing init_mlp_params."

    m = int(getattr(gen_cfg, "n_generators"))
    hidden_tau  = int(getattr(gen_cfg, "hidden_tau", 32))
    hidden_xi   = int(getattr(gen_cfg, "hidden_xi", 64))
    hidden_beta = int(getattr(gen_cfg, "hidden_beta", 64))

    # IMPORTANT: set input dims to match generator nets in the earlier cell.
    in_tau = 1
    in_xi  = 3   # <-- 2D case: take (t,x,y)
    in_beta = 3  # <-- 2D case: take (t,x,y)

    out_tau = 1
    out_xi  = 2  # <-- 2D vector field in space
    out_beta = 1

    keys = jax.random.split(key, 3 * m)
    params_tau_list, params_xi_list, params_beta_list = [], [], []

    for i in range(m):
        k_tau  = keys[3 * i]
        k_xi   = keys[3 * i + 1]
        k_beta = keys[3 * i + 2]

        params_tau  = init_mlp_params(k_tau,  sizes=[in_tau,  hidden_tau,  hidden_tau,  out_tau])
        params_xi   = init_mlp_params(k_xi,   sizes=[in_xi,   hidden_xi,   hidden_xi,   out_xi])
        params_beta = init_mlp_params(k_beta, sizes=[in_beta, hidden_beta, hidden_beta, out_beta])

        params_tau_list.append(params_tau)
        params_xi_list.append(params_xi)
        params_beta_list.append(params_beta)

    return {"tau": params_tau_list, "xi": params_xi_list, "beta": params_beta_list}

if _has_integer_leaves(params_gen):
    key_main, key_gen = jax.random.split(key_main)
    params_gen = _reinit_params_gen_2d(key_gen, gen_cfg)

# ---------- optimizer ----------
optimizer_gen = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(gen_train_cfg.lr),
)
opt_state_gen = optimizer_gen.init(params_gen)

# ---------- minibatch sampler ----------
rng_np_gen = np.random.default_rng(42)

def sample_tx_batch(batch_size: int):
    if batch_size >= N_tx:
        idx = np.arange(N_tx)
    else:
        idx = rng_np_gen.choice(N_tx, size=batch_size, replace=False)
    return jnp.asarray(TX_gen_np[idx], dtype=jnp.float64)  # (B,3)

# ---------- one JIT-ed training step ----------
@jax.jit
def train_step(params_gen, opt_state_gen, tx_batch, key):
    (loss_val, aux), grads = jax.value_and_grad(
        lambda p: master_loss(p, tx_batch, key=key),
        has_aux=True
    )(params_gen)

    updates, opt_state_new = optimizer_gen.update(grads, opt_state_gen, params_gen)
    params_new = optax.apply_updates(params_gen, updates)
    return params_new, opt_state_new, loss_val, aux

# ---------- helpers for logging ----------
LOSS_KEYS = ["L1", "L2", "L3", "L4", "L5", "L6", "L7"]

def _safe_float(x):
    return float(jnp.asarray(x))

def format_log(step, total_loss, aux):
    parts = [
        f"step {step:5d}/{gen_train_cfg.steps}",
        f"total={_safe_float(total_loss):.6e}",
    ]
    for k in LOSS_KEYS:
        if (k in aux) and isinstance(aux[k], dict) and ("loss" in aux[k]):
            parts.append(f"{k}={_safe_float(aux[k]['loss']):.3e}")
    if "weight_decay" in aux:
        parts.append(f"wd={_safe_float(aux['weight_decay']):.3e}")
    return " | ".join(parts)

# ---------- training loop ----------
loss_history_gen = []
loss_hist = {k: [] for k in LOSS_KEYS}
wd_history = []

key_main, key_train = jax.random.split(key_main, 2)

print(f"Starting 2D generator training for {gen_train_cfg.steps} steps "
      f"with batch_size={gen_train_cfg.batch_size}, lr={gen_train_cfg.lr}")


for step in range(1, gen_train_cfg.steps + 1):
    tx_batch = sample_tx_batch(gen_train_cfg.batch_size)  # (B,3) = [t,x,y]
    key_train, key_step = jax.random.split(key_train)

    #params_gen, opt_state_gen, loss_val, aux = train_step(params_gen, opt_state_gen, tx_batch, key_step)
    params_gen, opt_state_gen, loss_val, aux = train_step(params_gen, opt_state_gen, tx_batch, key_step)
    loss_val = jax.block_until_ready(loss_val)

    loss_history_gen.append(_safe_float(loss_val))

    for k in LOSS_KEYS:
        if (k in aux) and isinstance(aux[k], dict) and ("loss" in aux[k]):
            loss_hist[k].append(_safe_float(aux[k]["loss"]))
        else:
            loss_hist[k].append(np.nan)

    wd_history.append(_safe_float(aux.get("weight_decay", 0.0)))

    if (step % gen_train_cfg.print_every == 0) or (step == 1) or (step == gen_train_cfg.steps):
        print(format_log(step, loss_val, aux))

print("\n2D generator training finished.")

globals().update({
    "params_gen": params_gen,
    "opt_state_gen": opt_state_gen,
    "loss_history_gen": loss_history_gen,
    "loss_hist": loss_hist,          # dict: keys L1..L7 -> list
    "wd_history": wd_history,
    "gen_train_cfg": gen_train_cfg,
})


#SDE-sym. Evaluations

## Heat maps

In [ ]:
# === Evaluation 1: visualize learned generators (τ_i and ξ_i) — 2D-safe (expects y) ===

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

m = int(gen_cfg.n_generators)

# -------- user-adjustable grid settings --------------------------------------
t_min = float(t_surr.min())
t_max = float(t_surr.max())
x_min = float(x_surr.min())
x_max = float(x_surr.max())

# y-domain if available (2D notebooks usually have y_surr)
if "y_surr" in globals():
    y_min = float(y_surr.min())
    y_max = float(y_surr.max())
else:
    y_min, y_max = 0.0, 0.0

Nt_line = 200   # for τ(t) 1D plots
Nt = 50         # for ξ heatmaps (time resolution)
Nx = 50         # for ξ heatmaps (space resolution)

# For heatmaps: we plot ξ(t,x, y=y_slice) as a (t,x) slice
y_slice = 0.0
if y_min != y_max:
    y_slice = float(np.clip(y_slice, y_min, y_max))

print(f"Using t in [{t_min:.3f}, {t_max:.3f}], x in [{x_min:.3f}, {x_max:.3f}], y in [{y_min:.3f}, {y_max:.3f}]")
print(f"Heatmap grid: Nt={Nt}, Nx={Nx}, using y_slice={y_slice:.3f}")

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x, y):
    out = eval_generators_jit(params_gen, t, x, y)
    if not isinstance(out, (tuple, list)) or len(out) < 2:
        raise ValueError(f"eval_generators_jit must return (tau, xi, ...) but got: {type(out)}")
    tau, xi = out[0], out[1]
    return jnp.asarray(tau), jnp.asarray(xi)

# -------- 1D τ_i(t) curves ---------------------------------------------------
t_line = jnp.linspace(t_min, t_max, Nt_line)
x0 = jnp.zeros_like(t_line)
y0 = jnp.zeros_like(t_line)

tau_vals_line, _ = _eval_tau_xi(params_gen, t_line, x0, y0)  # (m, Nt_line)

plt.figure(figsize=(7, 2.5 * m))
for i in range(m):
    plt.subplot(m, 1, i + 1)
    plt.plot(np.asarray(t_line), np.asarray(tau_vals_line[i]), lw=2)
    plt.xlabel("t")
    plt.ylabel(rf"$\tau_{i+1}$")
    plt.title(rf"Generator {i+1}: $\tau_{i+1}(t, x=0, y=0)$")
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------- ξ_i(t,x) heatmaps (slice at fixed y=y_slice) -----------------------
t_eval = jnp.linspace(t_min, t_max, Nt)
x_eval = jnp.linspace(x_min, x_max, Nx)
TT, XX = jnp.meshgrid(t_eval, x_eval, indexing="ij")  # (Nt,Nx)

t_flat = TT.reshape(-1)
x_flat = XX.reshape(-1)
y_flat = jnp.full_like(t_flat, fill_value=jnp.asarray(y_slice, dtype=t_flat.dtype))

_, xi_vals_flat = _eval_tau_xi(params_gen, t_flat, x_flat, y_flat)

# xi_vals_flat expected:
#   (m, B, 2) for 2D xi
#   (m, B)    for 1D legacy
if xi_vals_flat.ndim == 2:
    xi_x_flat = xi_vals_flat
    xi_y_flat = None
elif xi_vals_flat.ndim == 3 and xi_vals_flat.shape[-1] == 2:
    xi_x_flat = xi_vals_flat[..., 0]
    xi_y_flat = xi_vals_flat[..., 1]
else:
    raise ValueError(f"Unexpected xi shape from eval_generators_jit: {xi_vals_flat.shape}")

xi_x_grids = xi_x_flat.reshape(m, Nt, Nx)
xi_y_grids = None if (xi_y_flat is None) else xi_y_flat.reshape(m, Nt, Nx)

# Plot: m rows, 1 or 2 columns depending on whether xi has y-component
ncols = 1 if xi_y_grids is None else 2
fig, axes = plt.subplots(m, ncols, figsize=(8 * ncols, 3 * m), constrained_layout=True)

# Normalize axes indexing
if m == 1 and ncols == 1:
    axes = np.array([[axes]])
elif m == 1 and ncols == 2:
    axes = np.array([axes])
elif m > 1 and ncols == 1:
    axes = np.array([[ax] for ax in axes])

for i in range(m):
    ax = axes[i, 0]
    im = ax.imshow(
        np.asarray(xi_x_grids[i]),
        origin="lower",
        extent=(float(x_eval.min()), float(x_eval.max()), float(t_eval.min()), float(t_eval.max())),
        aspect="auto"
    )
    ax.set_title(rf"Gen {i+1}: $\xi^x_{i+1}(t,x,y={y_slice:.2f})$")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    fig.colorbar(im, ax=ax, shrink=0.9)

    if ncols == 2:
        ax2 = axes[i, 1]
        im2 = ax2.imshow(
            np.asarray(xi_y_grids[i]),
            origin="lower",
            extent=(float(x_eval.min()), float(x_eval.max()), float(t_eval.min()), float(t_eval.max())),
            aspect="auto"
        )
        ax2.set_title(rf"Gen {i+1}: $\xi^y_{i+1}(t,x,y={y_slice:.2f})$")
        ax2.set_xlabel("x")
        ax2.set_ylabel("t")
        fig.colorbar(im2, ax=ax2, shrink=0.9)

plt.show()


## Span Check

In [ ]:
# === Evaluation 2: principal angles (m=3) against ground truth basis — 2D-safe (expects y) ===

import jax
import jax.numpy as jnp
import numpy as np

assert int(gen_cfg.n_generators) == 3, (
    f"Expected gen_cfg.n_generators == 3 for this evaluation, got {gen_cfg.n_generators}"
)

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x, y):
    out = eval_generators_jit(params_gen, t, x, y)
    if not isinstance(out, (tuple, list)) or len(out) < 2:
        raise ValueError(f"eval_generators_jit must return (tau, xi, ...) but got: {type(out)}")
    tau, xi = out[0], out[1]
    return jnp.asarray(tau), jnp.asarray(xi)

# -------- safe numeric k getter (never crashes; handles strings like 'L7' / None) ----------
def _safe_float(val, fallback: float = 1.0) -> float:
    # Always returns a float.
    if val is None:
        return float(fallback)
    try:
        if isinstance(val, (np.ndarray, jnp.ndarray)) and getattr(val, "shape", None) == ():
            val = val.item()
        return float(val)
    except Exception:
        return float(fallback)

# Prefer k_fp if numeric; else k; else 1.0
k_fp_raw = globals().get("k_fp", None)
k_raw    = globals().get("k", None)

k_fp_num = _safe_float(k_fp_raw, fallback=np.nan)
k_num    = _safe_float(k_raw,    fallback=np.nan)

if np.isfinite(k_fp_num):
    k_val = float(k_fp_num)
elif np.isfinite(k_num):
    k_val = float(k_num)
else:
    k_val = 1.0

k = jnp.asarray(k_val, dtype=jnp.float64)
print("Using k =", float(k_val))

# -------- user-adjustable evaluation grid ------------------------------------
t_min_eval = 0.0
t_max_eval = 2.0
x_min_eval = -1.0
x_max_eval = 1.0

# EDITABLE y-range (no y_surr dependency)
y_min_eval = -1.0
y_max_eval = 1.0

Nt_eval = 30
Nx_eval = 25
Ny_eval = 25

print(
    f"Principal-angle grid: t in [{t_min_eval:.3f}, {t_max_eval:.3f}], "
    f"x in [{x_min_eval:.3f}, {x_max_eval:.3f}], "
    f"y in [{y_min_eval:.3f}, {y_max_eval:.3f}], "
    f"Nt={Nt_eval}, Nx={Nx_eval}, Ny={Ny_eval}"
)

t_eval = jnp.linspace(t_min_eval, t_max_eval, Nt_eval)
x_eval = jnp.linspace(x_min_eval, x_max_eval, Nx_eval)
y_eval = jnp.linspace(y_min_eval, y_max_eval, Ny_eval)

TT, XX, YY = jnp.meshgrid(t_eval, x_eval, y_eval, indexing="ij")  # (Nt, Nx, Ny)

t_flat = TT.reshape(-1)
x_flat = XX.reshape(-1)
y_flat = YY.reshape(-1)
B = int(t_flat.shape[0])
print("Total evaluation points B =", B)

# -------- learned generators --------------------------------------------------
tau_learn, xi_learn = _eval_tau_xi(params_gen, t_flat, x_flat, y_flat)
# Expected:
#   tau_learn: (3, B)
#   xi_learn : (3, B, 2) ideally

# Normalize tau to (m,B)
if tau_learn.ndim == 1:
    # (B,) -> assume single generator; not expected here but keep safe
    tau_learn = tau_learn[None, :]
elif tau_learn.ndim == 2:
    pass
else:
    raise ValueError(f"Unexpected tau_learn shape: {tau_learn.shape}")

# Normalize xi to (m,B,d)
if xi_learn.ndim == 2:
    xi_comp = xi_learn[..., None]          # (m,B,1)
elif xi_learn.ndim == 3:
    xi_comp = xi_learn                     # (m,B,d)
else:
    raise ValueError(f"Unexpected xi_learn shape: {xi_learn.shape}")

m = 3
d = int(xi_comp.shape[-1])

def _stack_vec(tau_i, xi_i):
    # tau_i: (B,), xi_i: (B,d)
    return jnp.concatenate([tau_i.reshape(-1), xi_i.reshape(-1)], axis=0)  # (B + B*d,)

V = jnp.stack([_stack_vec(tau_learn[i], xi_comp[i]) for i in range(m)], axis=1)  # (B*(1+d), 3)

# -------- ground-truth generators (v1, v3, v5) -------------------------------
one  = jnp.ones_like(t_flat)
zero = jnp.zeros_like(t_flat)

# v1 = ∂t
w1_tau = one
w1_xi  = jnp.zeros((B, d), dtype=t_flat.dtype)
w1_vec = _stack_vec(w1_tau, w1_xi)

# v3 = e^{-k^2 t}(k^{-2}∂x - ∂y)
e = jnp.exp(-(k**2) * t_flat)
w3_tau = zero
w3_xi  = jnp.zeros((B, d), dtype=t_flat.dtype)
w3_xi  = w3_xi.at[:, 0].set(e * (k**-2) * one)     # xi_x
if d >= 2:
    w3_xi = w3_xi.at[:, 1].set(-e * one)           # xi_y
w3_vec = _stack_vec(w3_tau, w3_xi)

# v5 = ∂x
w5_tau = zero
w5_xi  = jnp.zeros((B, d), dtype=t_flat.dtype)
w5_xi  = w5_xi.at[:, 0].set(one)
w5_vec = _stack_vec(w5_tau, w5_xi)

W = jnp.stack([w1_vec, w3_vec, w5_vec], axis=1)  # (B*(1+d), 3)

# -------- principal angles ----------------------------------------------------
def principal_angles(V, W):
    Q1, _ = jnp.linalg.qr(V, mode="reduced")
    Q2, _ = jnp.linalg.qr(W, mode="reduced")
    s = jnp.linalg.svd(Q1.T @ Q2, compute_uv=False)
    s = jnp.clip(s, -1.0, 1.0)
    return jnp.sort(jnp.arccos(s))

angles_rad = principal_angles(V, W)
angles_deg = angles_rad * (180.0 / jnp.pi)

print("\nPrincipal angles between learned span{X_i} and ground-truth span{v1, v3, v5}:")
for idx, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
    print(f"  angle {idx}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

def cos_sim(a, b):
    return float((a @ b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-12))

print("\nPairwise cosine similarities (learned i vs ground-truth j):")
for i in range(3):
    for j in range(3):
        print(f"  <X_{i+1}, v_{j+1}> = {cos_sim(V[:, i], W[:, j]):.4f}")


In [ ]:
# === Evaluation 2b: stronger span check (block-balanced + best-mixing residual) ===
# Ground truth (for m=3 check) — SDE symmetries:
#   v1 = ∂_t
#   v3 = e^{-k^2 t}(k^{-2}∂_x-∂_y)
#   v5 = ∂_x
#
# Compatible with upstream: uses eval_generators_jit(params_gen, t, x) and ignores extra returns.

import jax
import jax.numpy as jnp
import numpy as np

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x):
    out = eval_generators_jit(params_gen, t, x)
    if not isinstance(out, (tuple, list)) or len(out) < 2:
        raise ValueError("eval_generators_jit must return (tau, xi, ...) with at least 2 outputs.")
    return out[0], out[1]

# -------- helper: per-generator block balancing (tau/xi comparable energy) ----
def _stack_balanced(tau, xi, eps=1e-12):
    """
    tau: (m, B)
    xi : (m, B) or (m, B, 2)
    Returns V: (D, m) where each column i has tau and xi separately standardized to unit RMS.
    """
    if xi.ndim == 2:
        # 1D xi
        tau_rms = jnp.sqrt(jnp.mean(tau**2, axis=1, keepdims=True) + eps)
        xi_rms  = jnp.sqrt(jnp.mean(xi**2,  axis=1, keepdims=True) + eps)
        tau_s = tau / tau_rms
        xi_s  = xi  / xi_rms
        V = jnp.concatenate([tau_s, xi_s], axis=1).T  # (2B, m)
        return V
    else:
        # 2D xi
        B = tau.shape[1]
        xi_flat = xi.reshape(xi.shape[0], B * xi.shape[2])  # (m, 2B)
        tau_rms = jnp.sqrt(jnp.mean(tau**2,     axis=1, keepdims=True) + eps)
        xi_rms  = jnp.sqrt(jnp.mean(xi_flat**2, axis=1, keepdims=True) + eps)
        tau_s = tau / tau_rms
        xi_s  = xi_flat / xi_rms
        V = jnp.concatenate([tau_s, xi_s], axis=1).T  # (3B, m)
        return V

# -------- helper: principal angles between column spans -----------------------
def principal_angles(V, W):
    Q1, _ = jnp.linalg.qr(V, mode="reduced")
    Q2, _ = jnp.linalg.qr(W, mode="reduced")
    s = jnp.linalg.svd(Q1.T @ Q2, compute_uv=False)
    s = jnp.clip(s, -1.0, 1.0)
    return jnp.sort(jnp.arccos(s))

# -------- helper: best-mixing residual ---------------------------------------
def best_mixing_residual(V, W, ridge=1e-10):
    k = W.shape[1]
    WT_W = W.T @ W
    A = jnp.linalg.solve(WT_W + ridge * jnp.eye(k, dtype=W.dtype), W.T @ V)
    Vhat = W @ A
    rel = jnp.linalg.norm(V - Vhat) / (jnp.linalg.norm(V) + 1e-12)
    return rel, A

# -------- choose evaluation points: prefer empirical TX_gen if available -------
USE_TX_GEN = True
B_max = 5000  # cap for speed

if USE_TX_GEN and ("TX_gen" in globals()):
    TX = jnp.asarray(TX_gen, dtype=jnp.float64)
    TX = TX[jnp.isfinite(TX).all(axis=1)]
    if TX.shape[0] == 0:
        raise ValueError("TX_gen is empty after filtering non-finites.")
    B = int(min(B_max, TX.shape[0]))
    t_flat = TX[:B, 0]
    x_flat = TX[:B, 1]
    print(f"[span-check] Using empirical TX_gen points: B={B}")
    print("t range:", float(t_flat.min()), float(t_flat.max()))
    print("x range:", float(x_flat.min()), float(x_flat.max()))

else:
    t_min_eval = float(t_surr.min()) if "t_surr" in globals() else 0.0
    t_max_eval = float(t_surr.max()) if "t_surr" in globals() else 5.0
    x_min_eval = float(x_surr.min()) if "x_surr" in globals() else -4.0
    x_max_eval = float(x_surr.max()) if "x_surr" in globals() else 6.0
    Nt_eval, Nx_eval = 40, 40
    t_eval = jnp.linspace(t_min_eval, t_max_eval, Nt_eval)
    x_eval = jnp.linspace(x_min_eval, x_max_eval, Nx_eval)
    TT, XX = jnp.meshgrid(t_eval, x_eval, indexing="ij")
    t_flat = TT.reshape(-1)
    x_flat = XX.reshape(-1)
    print("t range:", float(t_flat.min()), float(t_flat.max()))
    print("x range:", float(x_flat.min()), float(x_flat.max()))

    B = int(t_flat.shape[0])
    print(f"[span-check] Using uniform grid: B={B} on t∈[{t_min_eval},{t_max_eval}], x∈[{x_min_eval},{x_max_eval}]")

# -------- learned generators --------------------------------------------------
m = int(gen_cfg.n_generators)
tau_learn, xi_learn = _eval_tau_xi(params_gen, t_flat, x_flat)  # (m,B) and (m,B) or (m,B,2)

# -------- ground-truth subspace for m=3 check --------------------------------
one  = jnp.ones_like(t_flat)
zero = jnp.zeros_like(t_flat)
k = jnp.asarray(globals().get("k", globals().get("k_fp", 1.0)), dtype=t_flat.dtype)

# v1
w1_tau = one
w1_xi  = jnp.zeros_like(x_flat) if (xi_learn.ndim == 2) else jnp.zeros((B, 2), dtype=t_flat.dtype)

# v3
w3_tau = zero
if xi_learn.ndim == 2:
    w3_xi = jnp.exp(-(k**2) * t_flat) * (k**-2) * one
else:
    w3_xi = jnp.stack(
        [jnp.exp(-(k**2) * t_flat) * (k**-2) * one,
         -jnp.exp(-(k**2) * t_flat) * one],
        axis=-1
    )

# v5
w5_tau = zero
w5_xi  = one if (xi_learn.ndim == 2) else jnp.stack([one, zero], axis=-1)

W_tau = jnp.stack([w1_tau, w3_tau, w5_tau], axis=0)  # (3,B)
W_xi  = jnp.stack([w1_xi,  w3_xi,  w5_xi ], axis=0)  # (3,B) or (3,B,2)

# -------- build balanced stacked matrices V and W -----------------------------
k_dim = 3
if m < k_dim:
    raise ValueError(f"Need at least {k_dim} learned generators for this check; got m={m}.")

V_bal = _stack_balanced(tau_learn[:k_dim], xi_learn[:k_dim])  # (D, k)
W_bal = _stack_balanced(W_tau, W_xi)                          # (D, k)

# Also compute the "raw" (unbalanced) version for comparison
if xi_learn.ndim == 2:
    V_raw = jnp.concatenate([tau_learn[:k_dim], xi_learn[:k_dim]], axis=1).T  # (2B,k)
    W_raw = jnp.concatenate([W_tau, W_xi], axis=1).T                          # (2B,k)
else:
    B_here = tau_learn.shape[1]
    V_raw = jnp.concatenate([tau_learn[:k_dim], xi_learn[:k_dim].reshape(k_dim, 2*B_here)], axis=1).T  # (3B,k)
    W_raw = jnp.concatenate([W_tau, W_xi.reshape(k_dim, 2*B_here)], axis=1).T                            # (3B,k)

# -------- principal angles (raw vs balanced) ---------------------------------
angles_raw = principal_angles(V_raw, W_raw)
angles_bal = principal_angles(V_bal, W_bal)

def _deg(x): return x * (180.0 / jnp.pi)

print("\nPrincipal angles (RAW stacking):")
for i, a in enumerate(angles_raw, 1):
    print(f"  angle {i}: {float(a):.6f} rad = {float(_deg(a)):.4f} deg")

print("\nPrincipal angles (BALANCED tau/xi per generator):")
for i, a in enumerate(angles_bal, 1):
    print(f"  angle {i}: {float(a):.6f} rad = {float(_deg(a)):.4f} deg")

# -------- best-mixing residual (raw vs balanced) ------------------------------
rel_raw, A_raw = best_mixing_residual(V_raw, W_raw)
rel_bal, A_bal = best_mixing_residual(V_bal, W_bal)

print("\nBest-mixing relative residuals (lower is better):")
print(f"  RAW:      ||V - W A*|| / ||V|| = {float(rel_raw):.6e}")
print(f"  BALANCED: ||V - W A*|| / ||V|| = {float(rel_bal):.6e}")

print("\nBest-mixing matrix A* (BALANCED), columns correspond to learned generators in V:")
print(np.asarray(A_bal))

# -------- optional: pairwise cosines after balancing --------------------------
def cos_sim(a, b):
    return float((a @ b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-12))

print("\nPairwise cosine similarities using BALANCED stacked columns:")
for i in range(k_dim):
    for j in range(k_dim):
        cij = cos_sim(V_bal[:, i], W_bal[:, j])
        print(f"  <X_{i+1}, v_{j+1}> = {cij:.4f}")
